# 법인카드 이상거래 탐지 — 비지도 학습 파이프라인 통합본

**Tiger.inc · Risk Review Agent 1단계(비지도 이상 탐지) 개발 기록 — 전처리 → 피처셋 실험 → 최종 test 평가**

이 노트북은 아래 3개 노트북을 실행 순서대로 하나로 합친 통합본이다. 각 Part는 **자체적으로 데이터를 로드**하므로
Part 단위로 독립 실행이 가능하다 (Part 2·3은 Part 1이 저장한 `processed` CSV를 읽는다).

| Part | 원본 노트북 | 내용 |
|---|---|---|
| **Part 1** | `전처리_v3_세그먼트플래그` | 결측 정규화, 날짜 기준 train/test 재분할, leak-free 피처 엔지니어링, 4-Tier 피처 체계, fold 분할, 저장 |
| **Part 2** | `모델링_v1_Tier0vs1_비교` | Tier0 vs Tier0+1 비교, leave-one-in ablation, 최종 피처셋(Tier0+`일시불할부구분코드`) 확정 |
| **Part 3** | `모델링_v2_최종test평가` | train 전체 최종 학습, test(2024) 단 1회 채점, 세그먼트 분석, 고정 임계값 검증, pseudo-test |
| **Part 4** | (신규 추가) | recall 목표별 필요 검토 비율, 점수 구간별 실측 이상거래 비율(보정 테이블) — 결과보고서 §5·§4 재현용 |

> 참고: `전처리_v2_가맹점제외`는 v3(세그먼트 플래그 추가)로 대체된 구버전이므로 이 통합본에 포함하지 않는다.

**파이프라인 핵심 결정 요약**
- **비지도 학습(Isolation Forest)** — `이상거래여부` 라벨은 카드사 부정사용 기준이지 정산담당자 기준이 아니므로
  (label bias) 지도학습 타깃으로 쓰지 않고, **평가(채점) 전용**으로만 사용한다. MVP 지도학습 미사용 원칙은
  요구사항명세서 §9.1 / FR-RR-04·08 / FR-RL-02와 일치한다.
- **날짜 기준 train/test 분할** (~2023 train / 2024 test) — 원본 무작위 분할은 test가 train 피처 계산을
  오염시키는 구조라 폐기.
- **가맹점 파생 피처 완전 제외** — 여러 카드가 공유하는 가맹점 특성상 시간축 정합성 문제가 분할 방식과 무관하게
  재발함을 확인. 가맹점 신호는 RAG(Chroma `case_history`)로 역할 이관.
- **최종 피처셋: Tier0(14개) + `일시불할부구분코드`(1개) = 15개** (그중 모델 직접 입력 13개, `거래일자`·`거래연월`은
  파생 계산용 식별자 — 원-핫 후 입력 행렬 24개 컬럼).
- **최종 test PR-AUC 0.5865** (베이스레이트 3.49%), 고정 임계값(train 97퍼센타일, score ≥ 0.0620) 적용 시
  test 분류 비율 3.29% / recall 0.550 / precision 0.583.
- **컷오프(top3%)는 잠정** — recall 분석 결과(Part 4) "이상거래를 놓치면 안 된다"는 요구 수준에 못 미쳐 재검토 필요.

**섹션 번호 규칙**: 각 Part 내부의 절 번호(`## 1.` 등)는 원본 노트북 번호를 그대로 유지한다. 본문에서 "5절",
"7-1절" 같은 상호 참조는 **같은 Part 안의 절**을 가리킨다.


---
# Part 1 — 전처리 (원본: `법인카드_이상거래_전처리_v3_세그먼트플래그.ipynb`)


# 법인카드 정산 자동화 · 이상거래 탐지 — 전처리 (Preprocessing)

**목적**: `법인카드_이상거래_EDA_v3.ipynb`에서 확인한 데이터 품질 이슈·데이터 누수 문제·Feature Engineering 결과를 반영해,
`train`/`test` 원본 CSV를 모델 학습에 바로 투입 가능한 형태로 정리한다.

**이 노트북에서 지키는 원칙 (모두 EDA_v3 근거, 3-1절은 `전처리_노트북_리뷰_통합본.md` 근거)**
- **Look-ahead 피처는 만들지 않는다.** EDA 9절의 `사용자평균사용액`/`가맹점평균금액`/`거래금액_Zscore`(카드·가맹점 전체 기간 통계)는
  진단용으로만 유효했고, 실제 파이프라인에는 9-1절의 **Expanding Window(시점 이전 데이터만 사용)** 버전만 사용한다.
- **콜드스타트 정보를 숨기지 않는다.** EDA 10-4절에서 확인했듯 `fillna(median)`으로 결측을 채우면 "이력 없음" 상태가
  "평범한 값"으로 위장된다. 이 노트북은 결측을 임의로 채우지 않고 `카드/가맹점 첫거래여부` 플래그로 남겨, 트리 기반
  모델(LightGBM/XGBoost 등, EDA 10절 권장)이 결측을 네이티브로 처리하도록 둔다.
- **`'_'` 플레이스홀더는 명시적으로 NaN 처리한다.** EDA 3절에서 확인했듯 `.isnull()`로 잡히지 않는 은닉 결측치다.
- **금액=0원, 전월 실적 음수 같은 특이값은 제거하지 않고 플래그로 남긴다.** EDA 3절 결론(0원 거래는 위험 신호 아님,
  전월 실적 음수는 오히려 위험 신호 후보)에 따라 임의 삭제 대신 플래그 피처로 판단을 모델에 넘긴다.
- **Train/Test는 날짜 기준으로 격리한다(3-1절).** AI Hub 원본 train/test 파일은 같은 모집단의 무작위 분할이라,
  시간 기반 피처를 `shift(1)`로 안전하게 계산해도 "test가 train 피처 계산에 개입하지 않아야 한다"는 격리 원칙은
  지켜지지 않는다. 그래서 원본 파일 경계를 버리고 `거래일자` 기준으로 다시 나눈다.

**⚠️ train/test 분할 방식에 대한 중요한 사실 확인 (그리고 왜 그대로 쓰지 않는가)**
`.data/train`(173.6만 건)과 `.data/test`(21.7만 건)는 **같은 모집단의 무작위 분할**이다(둘 다 2021-01-01~2024-12-31 동일 기간,
이상거래 비율도 3.687% vs 3.687%로 사실상 동일, test에 등장하는 카드 2,577개는 **전부** train에도 존재).
즉 **미래 시점 홀드아웃이 아니라 같은 시기의 무작위 샘플**이다.

처음에는 이 경계를 그대로 유지하되, 시간 기반 피처(최근7일사용횟수, 카드누적사용액, Expanding Window 통계)만
train+test를 합쳐서 시간순으로 계산한 뒤 원래 소속으로 재분리하는 방식으로 대응했다 — 각 피처가 "해당 거래 시점
이전 정보만" 쓰도록(shift(1)/rolling) 만들면 look-ahead 누수는 없다고 봤기 때문이다.

하지만 리뷰(`전처리_노트북_리뷰_통합본.md` 🟠)에서 지적했듯, 그것만으로는 부족하다: look-ahead(미래 시점 정보 사용)는
막아도, **train/test 오염**(평가용 test 데이터가 train 피처 계산에 개입하는 것)은 막지 못한다. 카드 A가 3월엔
test 파일에, 6월엔 train 파일에 있으면, 6월(train) 거래의 확장 피처에 3월(test) 거래 금액이 그대로 섞여 들어간다 —
미래를 보는 건 아니지만 평가셋이 학습 과정에 개입하는 것은 맞다.

그래서 이 노트북은 **원본 train/test 파일 구분을 버리고, `거래일자` 기준으로 다시 나눈다(3-1절)**. test가 항상
train보다 시간상 뒤에 있으므로 오염이 구조적으로 불가능해지고, "과거 이력으로 미래 거래를 판단"하는 실제 운영
시나리오와도 일치한다.

## 1. 라이브러리 & 데이터 로드

In [1]:
import pandas as pd
import numpy as np
import glob
import os
import warnings
import json

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)

RAW_DIR = '../.data'
TRAIN_DIR = os.path.join(RAW_DIR, 'train')
TEST_DIR = os.path.join(RAW_DIR, 'test')
FILE_PATTERN = '21-2_카드데이터_*.csv'
OUT_DIR = os.path.join(RAW_DIR, 'processed')
os.makedirs(OUT_DIR, exist_ok=True)


def load_transaction_data(data_dir, pattern=FILE_PATTERN):
    '''data_dir 내 csv 파일을 자동 인식하여 하나의 DataFrame으로 통합. (EDA_v3 2절과 동일한 로더)'''
    files = sorted(glob.glob(os.path.join(data_dir, pattern)))
    if not files:
        raise FileNotFoundError(f"'{data_dir}' 경로에서 '{pattern}' 패턴의 csv 파일을 찾지 못했습니다.")
    frames = [pd.read_csv(f, encoding='utf-8', low_memory=False) for f in files]
    return pd.concat(frames, ignore_index=True)


train_raw = load_transaction_data(TRAIN_DIR)
test_raw = load_transaction_data(TEST_DIR)
print(f"train_raw: {train_raw.shape}")
print(f"test_raw : {test_raw.shape}")

train_raw: (1735885, 32)
test_raw : (216986, 32)


In [2]:
# train/test가 정말 같은 모집단의 무작위 분할인지 재확인 (위 인트로에서 언급한 근거를 이 노트북 안에서도 재현).
# 이 사실 자체는 여전히 참이다 — 그래서 원본 경계 대신 날짜 기준으로 재분할한다(3-1절, 리뷰 통합본 🟠).
print(f"train 이상거래 비율: {train_raw['이상거래여부'].mean()*100:.3f}%")
print(f"test  이상거래 비율: {test_raw['이상거래여부'].mean()*100:.3f}%")

train_cards = set(train_raw['카드KEY'])
test_cards = set(test_raw['카드KEY'])
card_overlap = len(test_cards & train_cards)
print(f"\ntest 카드 {len(test_cards):,}개 중 train에도 존재하는 카드: {card_overlap:,}개 "
      f"({card_overlap/len(test_cards)*100:.1f}%)")
print(f"train 기간: {train_raw['승인일자'].min()} ~ {train_raw['승인일자'].max()}")
print(f"test  기간: {test_raw['승인일자'].min()} ~ {test_raw['승인일자'].max()}")

# 날짜 기준 재분할이 안전한지 확인하기 위한 연도별 이상거래 비율(리뷰 통합본 🟠 "진행 전 확인 필요" 1번).
# 특정 연도만 뚝 떼어 test로 쓸 때 이상거래 비율이 크게 튀지 않아야 그 연도를 test로 써도 대표성이 있다고 볼 수 있다.
_all_raw = pd.concat([train_raw, test_raw], ignore_index=True)
_all_raw['_연도'] = _all_raw['승인일자'].astype(str).str[:4]
_yearly = _all_raw.groupby('_연도')['이상거래여부'].agg(건수='size', 이상거래비율='mean')
_yearly['이상거래비율'] = (_yearly['이상거래비율'] * 100).round(3)
print("\n연도별 건수/이상거래 비율(%):")
print(_yearly)

train 이상거래 비율: 3.687%
test  이상거래 비율: 3.687%

test 카드 2,577개 중 train에도 존재하는 카드: 2,577개 (100.0%)
train 기간: 20210101 ~ 20241231
test  기간: 20210101 ~ 20241231

연도별 건수/이상거래 비율(%):
          건수  이상거래비율
_연도                 
2021  506873   3.689
2022  492726   4.122
2023  483370   3.435
2024  469902   3.489


In [3]:
# 원본 파일(train/test) 소속을 태깅해둔다. 이제 이건 최종 분할 기준이 아니다(3-1절에서 거래일자 기준으로
# '데이터셋구분'을 새로 만든다) — 정렬 시 승인SEQ 충돌을 결정론적으로 깨기 위한 타이브레이커 + 출처 기록용으로만 쓴다.
train_raw['원본파일_구분'] = 'train'
test_raw['원본파일_구분'] = 'test'
df = pd.concat([train_raw, test_raw], ignore_index=True)
print("결합 데이터 shape:", df.shape)

결합 데이터 shape: (1952871, 33)


## 2. 결측치 정규화 (`'_'` → NaN)

EDA_v3 3절에서 확인했듯 `남녀구분코드`, `개인법인구분코드_가맹점`, `가맹점형태구분코드`, `일시불할부구분코드`,
`가맹점승인업종코드`, `승인거래코드`, `승인발생경로코드` 등 문자열 컬럼에는 결측치(NaN)와 별도로 `'_'` 플레이스홀더가
쓰인다. `.isnull()`로는 잡히지 않으므로 명시적으로 NaN 처리한다.

In [4]:
underscore_cols = [
    col for col in df.columns
    if pd.api.types.is_object_dtype(df[col]) and (df[col] == '_').any()
]
print("'_' 플레이스홀더가 있는 컬럼:", underscore_cols)

before_na = df[underscore_cols].isna().sum()
df[underscore_cols] = df[underscore_cols].replace('_', np.nan)
after_na = df[underscore_cols].isna().sum()

pd.DataFrame({'처리 전 결측': before_na, "'_' → NaN 처리 후 결측": after_na})

'_' 플레이스홀더가 있는 컬럼: ['승인거래코드']


,처리 전 결측,'_' → NaN 처리 후 결측
승인거래코드,0,970


## 3. 날짜/시간 파생 변수 (EDA_v3 7·9절과 동일한 정의)

In [5]:
df['거래일자'] = pd.to_datetime(df['승인일자'], format='%Y%m%d')
df['거래연월'] = df['거래일자'].dt.to_period('M')
df['거래요일_한글'] = df['거래일자'].dt.dayofweek.map(
    {0: '월', 1: '화', 2: '수', 3: '목', 4: '금', 5: '토', 6: '일'}
)


def categorize_hour(h):
    if 0 <= h < 6:
        return '심야(00-05)'
    elif 6 <= h < 12:
        return '오전(06-11)'
    elif 12 <= h < 18:
        return '오후(12-17)'
    else:
        return '저녁(18-23)'


# 승인시간대(원본 0~23시)와 시간대구간(파생 4구간)을 둘 다 남긴다 — 트리 모델엔 중복이어도 해가 없고,
# 향후 선형/거리 기반 모델에서 더 세밀한 시각 신호가 필요할 수 있어 의도적으로 유지한다(리뷰 통합본 경미 사항).
df['시간대구간'] = df['승인시간대'].apply(categorize_hour)
df['월말여부'] = df['거래일자'].dt.day >= 25

df[['거래일자', '거래연월', '거래요일_한글', '시간대구간', '월말여부']].head()

,거래일자,거래연월,거래요일_한글,시간대구간,월말여부
0,2021-02-07,2021-02,일,오전(06-11),False
1,2021-02-19,2021-02,금,오후(12-17),False
2,2021-01-11,2021-01,월,오후(12-17),False
3,2021-03-28,2021-03,일,오후(12-17),True
4,2021-03-05,2021-03,금,오후(12-17),False


## 3-1. Train/Test 재분할 (날짜 기준) — 리뷰 통합본 🟠 반영

원본 train/test 파일 구분(2절에서 재확인)은 버리고, 위에서 만든 `거래일자` 기준으로 다시 나눈다:
**~2023 → train, 2024 → test.** 컷오프보다 뒤(2024)를 test로 고정하면 test가 항상 train보다 시간상 뒤에 있으므로,
train 피처 계산에 test 정보가 섞여 들어가는 오염이 구조적으로 불가능해진다.

진행 전 확인한 사항(리뷰 통합본 "진행 전 확인 필요" 3가지):
1. **연도별 이상거래 비율 안정성** — 2절에서 확인: 2021 3.689% / 2022 4.122% / 2023 3.435% / 2024 3.489%. 2024만
   따로 떼도 전체 평균(3.687%)에서 크게 벗어나지 않는다.
2. **컷오프 단위(연도 vs 최근 N개월)** — "연도 단위"를 택했다. "최근 5개월(2024-08~12)"로 하면 원본 test
   비중(11.1%)과 더 비슷하게 맞출 수 있지만(실측 9.88%), 팀 문서·재현성 측면에서 "연도 단위"가 더 단순하고
   설명하기 쉽다. 그 대가로 test 비중이 원본(11.1%)보다 커진다(약 24%) — 최종 평가셋이 커지는 것이라 통계적으로는
   오히려 나쁘지 않다.
3. **9절 `StratifiedGroupKFold` 영향** — 그 fold는 아래 7절에서 `train_df`가 확정된 *이후*에 새로 계산되므로
   이번 재분할과 독립적이다.

In [6]:
CUTOFF_DATE = '2024-01-01'  # ~2023 = train, 2024 = test (위 설명 근거)
df['데이터셋구분'] = np.where(df['거래일자'] < CUTOFF_DATE, 'train', 'test')

_split_summary = df.groupby('데이터셋구분')['이상거래여부'].agg(건수='size', 이상거래비율='mean')
_split_summary['이상거래비율'] = (_split_summary['이상거래비율'] * 100).round(3)
_split_summary['비중(%)'] = (_split_summary['건수'] / len(df) * 100).round(2)
print(_split_summary)

             건수  이상거래비율  비중(%)
데이터셋구분                        
test     469902   3.489  24.06
train   1482969   3.750  75.94


## 4. 이상값·특이값 플래그 (EDA_v3 3절 결론 반영)

셋 다 **행을 삭제하지 않고** 플래그 컬럼으로만 남긴다 — EDA에서 확인했듯 셋 다 단순 오류로 단정할 근거가 부족하거나
(0원 거래는 오히려 이상거래 비율이 평균보다 낮음), 오히려 위험 신호일 수 있어서(전월 실적 음수) 삭제가 정보 손실이다.

In [7]:
# (1) 통합승인금액 = 0원: EDA 3절에서 이상거래 비율 0.52%(전체 평균 3.69%)로 오히려 낮음을 확인 → 취소/인증성 거래로 추정
df['취소성거래_추정'] = (df['통합승인금액'] == 0).astype(int)

# (2) 전월 실적 음수: EDA 3절에서 이상거래 비율이 평균의 최대 3배 이상으로 높음을 확인 → 위험 신호 후보 플래그로 보존
df['전월매출건수_음수여부'] = (df['전월_매출건수'] < 0).astype(int)
df['전월매출금액_음수여부'] = (df['전월_매출금액'] < 0).astype(int)

# (3) 카드이용한도금액: EDA 3절에서 법인카드 거래의 약 98%가 0원(미기재)으로 기록됨을 확인 → 값 자체보다 "미기재 추정" 여부가 더 신뢰할 수 있는 신호
df['법인카드_한도미기재추정'] = (
    (df['개인법인구분코드_회원'] == 2.0) & (df['카드이용한도금액'] == 0)
).astype(int)

# 카드이용한도금액_구간: pd.Categorical(values, categories=[...])는 "구간(bin)"이 아니라 정확히 일치하는 값만
# 카테고리로 인정한다(pd.cut과 다름). 컬럼명만 보면 binning처럼 보이지만, train+test 전체에서
# 카드이용한도금액이 실제로 {0, 1,000,000, 5,000,000, 10,000,000} 4개 값만 가짐을 확인했으므로(리뷰 통합본 버그2 —
# "구간화가 안 됨"처럼 보이지만 애초에 구간이 아니라 정확값 매칭이 맞는 케이스) pd.cut 대신 정확값 매칭을
# 그대로 쓴다. 값 분포가 바뀌면(예: 다른 한도 금액이 섞여 들어오면) 아래 assert가 걸려 알 수 있다.
_expected_limit_values = {0, 1_000_000, 5_000_000, 10_000_000}
_actual_limit_values = set(df['카드이용한도금액'].dropna().unique())
assert _actual_limit_values <= _expected_limit_values, (
    f"카드이용한도금액에 예상치 못한 값이 있습니다: {_actual_limit_values - _expected_limit_values}"
)
df['카드이용한도금액_구간'] = pd.Categorical(
    df['카드이용한도금액'],
    categories=[0, 1_000_000, 5_000_000, 10_000_000],
    ordered=True,
)

flag_cols = ['취소성거래_추정', '전월매출건수_음수여부', '전월매출금액_음수여부', '법인카드_한도미기재추정']
(df[flag_cols].mean() * 100).round(3).rename('비율(%)')

취소성거래_추정        0.147
전월매출건수_음수여부     0.025
전월매출금액_음수여부     0.054
법인카드_한도미기재추정    9.514
Name: 비율(%), dtype: float64

## 5. 시점 안전(Leak-free) Feature Engineering — EDA_v3 9절·9-1절·10-4절 근거

`최근7일사용횟수`·`카드누적사용액`은 원래도 시점 안전(rolling/cumsum, 해당 거래까지의 정보만 사용)했으므로 그대로 가져온다.
`사용자평균사용액`·`거래금액_Zscore`는 EDA 9절에서 전체 기간(look-ahead) 버전으로 만들었다가 9-1절에서
데이터 누수를 확인했으므로, **이 노트북에서는 Expanding Window(shift(1)) 버전만** 만든다. 첫 거래(과거 이력 없음)는
EDA 10-4절 교훈대로 `fillna`로 채우지 않고 `카드첫거래여부` 플래그와 결측(NaN)으로 그대로 남긴다.

**[변경] 가맹점 관련 피처(`가맹점평균금액_확장`, `가맹점첫거래여부`)는 이 노트북에서 만들지 않는다.**
`전처리_노트북_리뷰_통합본.md` 버그1에서 지적된 look-ahead 누수를 카드 기준 분할과 무관하게 원천 차단하기 위해,
그리고 가맹점 기준 통계는 프로덕션에서도 별도 RAG 파이프라인(Chroma `case_history`)이 담당하도록 역할을 분리하기로
팀에서 결정했다. 카드 단위 피처(`카드KEY`로만 그룹핑하는 피처)는 카드 하나가 항상 단일 정체성을 가지므로 이 결정과
무관하게 안전하며 그대로 유지한다.


In [8]:
# 카드별 시간순 정렬. 승인SEQ는 파일(분기)별로 채번되어 train/test 결합 후에는 완전히 유일하지 않을 수 있으므로
# 원본파일_구분까지 정렬 키에 포함해 그나마 결정론적으로 순서를 고정한다(실제로는 (카드KEY,승인일자,승인SEQ) 조합이
# train 내에서 이미 유일함을 확인했다 — 완전한 동시성 해소는 원본에 타임스탬프가 없어 불가능. '데이터셋구분'은
# 3-1절에서 만든 신규 train/test 분할 기준이라 정렬 타이브레이커로는 '원본파일_구분'을 대신 쓴다).
df = df.sort_values(['카드KEY', '거래일자', '승인SEQ', '원본파일_구분'])

# --- 시점 안전 롤링/누적 피처 (EDA_v3 9절과 동일한 방식, 원래도 leak-free) ---
# groupby+rolling 결과의 인덱스는 (카드KEY, 거래일자) 멀티인덱스라 원래 행 순서로 바로 대입할 수 없으므로,
# 정렬 전 위치(orig_positions)를 저장해뒀다가 위치 기반으로 안전하게 복원한다.

def add_rolling_count(frame, key_col, date_col, window='7D', out_col='최근7일사용횟수'):
    t = frame[[key_col, date_col]].copy()
    t['_one'] = 1
    t_sorted = t.sort_values([key_col, date_col])
    orig_positions = t_sorted.index
    t_sorted = t_sorted.set_index(date_col)
    rolled = t_sorted.groupby(key_col)['_one'].rolling(window).sum()
    result = pd.Series(rolled.values, index=orig_positions).sort_index()
    frame[out_col] = result
    return frame


df = add_rolling_count(df, '카드KEY', '거래일자')
df['카드누적사용액'] = df.groupby('카드KEY')['통합승인금액'].cumsum()
# 주의: cumsum은 당일(해당 행) 거래 금액을 포함한다. 아래 `_확장`(shift(1)) 계열 피처들은 당일 거래를 제외한
# "그 이전까지의" 통계라 기준 시점이 다르다(리뷰 통합본 경미 사항 — 누수는 아니지만 혼동 방지용 기록).

# --- Expanding Window(leak-free) 피처 (EDA 9-1절) — 카드 단위만 ---
# df가 카드KEY 기준으로 이미 정렬돼 있으므로 groupby('카드KEY')는 그룹 내부가 곧 시간순이라 안전하다.
# [변경] 가맹점(가맹점KEY) 기준 확장 피처는 만들지 않는다 — 별도 정렬·reindex 없이는 look-ahead 누수가
# 발생하는 걸 리뷰에서 확인했고(버그1), 가맹점 신호는 RAG(case_history)로 역할을 옮기기로 했으므로
# 이 파이프라인에서는 아예 제거한다.
card_amt = df.groupby('카드KEY')['통합승인금액']
df['사용자평균사용액_확장'] = card_amt.transform(lambda s: s.expanding().mean().shift(1))
df['사용자표준편차_확장'] = card_amt.transform(lambda s: s.expanding().std().shift(1))

df['거래금액_Zscore_확장'] = (
    (df['통합승인금액'] - df['사용자평균사용액_확장'])
    / df['사용자표준편차_확장'].replace(0, np.nan)
)

# --- 콜드스타트 플래그 (EDA 10-4절 교훈: 결측을 채우지 말고 명시적으로 남긴다) ---
df['카드첫거래여부'] = df['사용자평균사용액_확장'].isna().astype(int)

df = df.sort_index()  # concat 직후의 원래 행 순서(=원본 train 파일 다음 원본 test 파일) 복원

ew_cols = ['최근7일사용횟수', '카드누적사용액', '사용자평균사용액_확장', '거래금액_Zscore_확장']
flag_cols2 = ['카드첫거래여부']
print("시점 안전 피처 결측 비율(%):")
print((df[ew_cols].isna().mean() * 100).round(2))
print("\n콜드스타트 플래그 비율(%):")
print((df[flag_cols2].mean() * 100).round(2))


시점 안전 피처 결측 비율(%):
최근7일사용횟수          0.00
카드누적사용액           0.00
사용자평균사용액_확장       0.13
거래금액_Zscore_확장    0.34
dtype: float64

콜드스타트 플래그 비율(%):
카드첫거래여부    0.13
dtype: float64


## 6. 범주형 변수 dtype 정리

EDA_v3 10절 결론(트리 기반 앙상블 권장)에 맞춰 원-핫 인코딩 대신 `category` dtype으로만 캐스팅한다.
LightGBM은 `category` dtype을 네이티브로 처리하고, XGBoost/scikit-learn 계열을 쓸 경우 `col.cat.codes`로
정수 인코딩하거나 `pd.get_dummies`를 나중에 적용하면 된다. 결측(NaN)은 임의로 채우지 않고 그대로 category의
"결측"으로 남긴다 — EDA 3절에서 확인했듯 `남녀구분코드`가 NaN(구 `'_'`)이라는 사실 자체가 법인카드 신호였기 때문이다.

In [9]:
CATEGORICAL_COLS = [
    '개인법인구분코드_회원', '국내해외여부', '남녀구분코드', '승인거래코드', '승인발생경로코드',
    '개인법인구분코드_가맹점', '가맹점여부_신규', '인터넷판매여부', '가맹점상태코드', '가맹점형태구분코드',
    '로고구분코드', '일시불할부구분코드', '카드구분코드', '가맹점누적매출금액_구간화', '연령',
    '가맹점광역시도코드', '가맹점승인업종코드', '시간대구간', '거래요일_한글',
]

for col in CATEGORICAL_COLS:
    df[col] = df[col].astype('category')

df[CATEGORICAL_COLS].dtypes

개인법인구분코드_회원      category
국내해외여부           category
남녀구분코드           category
승인거래코드           category
승인발생경로코드         category
개인법인구분코드_가맹점     category
가맹점여부_신규         category
인터넷판매여부          category
가맹점상태코드          category
가맹점형태구분코드        category
로고구분코드           category
일시불할부구분코드        category
카드구분코드           category
가맹점누적매출금액_구간화    category
연령               category
가맹점광역시도코드        category
가맹점승인업종코드        category
시간대구간            category
거래요일_한글          category
dtype: object

## 7. train/test 재분리(날짜 기준) 및 정합성 검증

In [10]:
train_df = df[df['데이터셋구분'] == 'train'].drop(columns=['데이터셋구분', '원본파일_구분']).reset_index(drop=True)
test_df = df[df['데이터셋구분'] == 'test'].drop(columns=['데이터셋구분', '원본파일_구분']).reset_index(drop=True)

print(f"train_df: {train_df.shape}")
print(f"test_df : {test_df.shape}")
assert len(train_df) + len(test_df) == len(train_raw) + len(test_raw), "재분리 후 전체 행 수가 원본과 다릅니다."

print(f"\ntrain 이상거래 비율: {train_df['이상거래여부'].mean()*100:.3f}%")
print(f"test  이상거래 비율: {test_df['이상거래여부'].mean()*100:.3f}%")

# [변경] 가맹점첫거래여부는 5절에서 더 이상 생성하지 않으므로 콜드스타트 비율은 카드 기준만 확인한다.
print("\ntrain 콜드스타트 비율(%):", (train_df[['카드첫거래여부']].mean() * 100).round(2).to_dict())
print("test  콜드스타트 비율(%):", (test_df[['카드첫거래여부']].mean() * 100).round(2).to_dict())

# 참고: 날짜 기준 재분할이라 카드 재사용률이 원본(100%)과 비슷하게 높게 유지되는 게 정상이다 — 이제는 "같은 카드가
# train/test에 걸쳐 있는 것" 자체가 문제가 아니라(오히려 실제 운영과 같은 상황: 같은 카드가 계속 쓰인다),
# 시점 안전 피처(shift(1)/rolling)가 항상 "그 시점 이전 정보만" 쓰도록 보장돼 있는지가 핵심이다(5절에서 보장).
train_cards2 = set(train_df['카드KEY'])
test_cards2 = set(test_df['카드KEY'])
overlap2 = len(train_cards2 & test_cards2)
print(f"\ntest 카드 {len(test_cards2):,}개 중 train에도 존재하는 카드: {overlap2:,}개 ({overlap2/len(test_cards2)*100:.1f}%)")


train_df: (1482969, 48)
test_df : (469902, 48)

train 이상거래 비율: 3.750%
test  이상거래 비율: 3.489%

train 콜드스타트 비율(%): {'카드첫거래여부': 0.17}
test  콜드스타트 비율(%): {'카드첫거래여부': 0.02}

test 카드 2,438개 중 train에도 존재하는 카드: 2,365개 (97.0%)


## 7-1. Test 카드 세그먼트 플래그 — 재사용 카드 vs 신규 카드

`test_df`의 이상탐지 성능은 전체 하나의 숫자로만 보면 "카드 이력이 풍부한 재사용 카드"와 "이력이 전혀 없는
신규 카드(콜드스타트)"의 성능이 뭉쳐서 가려진다. 카드 통계 피처(`사용자평균사용액_확장` 등)에 대한 모델의
의존도를 진단하고, 신규 카드 유입 상황(신규 입사자 법인카드 발급 등)에서의 실제 대응력을 별도로 확인하기 위해
`test_df`에 세그먼트 플래그를 미리 만들어 저장해둔다.

이 플래그는 **모델 학습 피처가 아니라 평가 전용 진단 컬럼**이므로 `feature_cols`(8절)에는 포함하지 않는다.


In [11]:
# train에 등장했던 카드 집합을 기준으로, test의 각 거래가 "재사용 카드"인지 "신규 카드(test 최초 등장)"인지 표시.
# 카드 통계 피처(사용자평균사용액_확장 등)에 대한 모델의 의존도를 평가 단계에서 나눠보기 위한 진단용 컬럼이며,
# 모델 학습에는 쓰지 않는다(8절 feature_cols에서 제외).
_seen_cards_in_train = set(train_df['카드KEY'])
test_df['카드_train노출여부'] = test_df['카드KEY'].isin(_seen_cards_in_train)

_seg_summary = test_df.groupby('카드_train노출여부').agg(
    거래건수=('카드KEY', 'size'),
    이상거래비율=('이상거래여부', 'mean'),
    카드첫거래비율=('카드첫거래여부', 'mean'),
)
_seg_summary['이상거래비율'] = (_seg_summary['이상거래비율'] * 100).round(3)
_seg_summary['카드첫거래비율'] = (_seg_summary['카드첫거래비율'] * 100).round(2)
print("test 세그먼트별 요약 (True=train에서 이미 본 카드, False=test 최초 등장 카드):")
print(_seg_summary)


test 세그먼트별 요약 (True=train에서 이미 본 카드, False=test 최초 등장 카드):
                거래건수  이상거래비율  카드첫거래비율
카드_train노출여부                         
False           4574   3.564      1.6
True          465328   3.488      0.0


## 8. 모델링에 사용할 피처 목록 정리 — 프로덕션 스키마 기준 4단계 Tier

원본 컬럼은 대부분 그대로 두되, `이상거래유형`·`이상거래설명`은 데이터셋에서 완전히 제거한다(아래 코드) —
세부유형은 카드사 자체 부정사용 분류 체계라 정산담당자 판단 기준과 무관하고(EDA 10-3절), 설명은 96% 결측인
고카디널리티 자유텍스트라 활용도가 낮다.

**[변경] 가맹점 관련 원본 속성 컬럼(가맹점광역시도코드 등, 기존 Tier 3)은 그대로 두되, 가맹점 기준 파생 피처
(`가맹점평균금액_확장`, `가맹점첫거래여부`)는 5절에서 아예 생성하지 않으므로 이 목록에도 없다.** 가맹점 신호는
이 지도학습/비지도 피처 파이프라인이 아니라 RAG(Chroma `case_history`) 쪽에서 다루기로 역할을 분리했다.

남은 컬럼은 "실제 운영 환경(기술명세서 §3.1)에서 이 피처를 계산할 수 있는 근거가 어디서 오는가"를 기준으로
4단계 Tier로 나눈다. 실제 `transactions` 테이블은 `(merchant, amount, ts, card_id, raw_payload JSONB)`뿐이라
이 AI Hub 데이터의 32개 컬럼이 그대로 오는 게 아니기 때문이다.

| Tier | 근거 | 프로덕션 가용성 |
|---|---|---|
| **Tier 0** | `transactions` 핵심 필드(금액·시각·카드ID)만으로 계산. 가맹점 파생 피처는 제외 | 거의 확실히 유지됨 |
| **Tier 1** | 카드/거래 정산 메타데이터(카드구분·한도·할부·승인경로 등). 법인카드 계정 관리에 필요해 카드사가 기본 제공할 가능성이 높음 | `raw_payload`에 포함될 가능성 높음(다만 §11.2 확정 전) |
| **Tier 2** | 사용자·부서·직급 등 조직 데이터 | 이 데이터셋엔 아예 없음 — `users`/`teams` 조인이 실제로 연동돼야 채워짐 |
| **Tier 3** | 가맹점 속성 전체 + 카드홀더 인적정보. 가맹점 속성은 MCC가 **post-MVP**로 미뤄져 있고 자체 카카오/웹검색 캐스케이드(§7-1)는 `{industry_code, industry_label, confidence, source}`라는 **다른 스키마**를 만들어서 이 원본 컬럼 값 자체는 어느 경로로도 재현이 안 됨. 인적정보(연령/성별)는 법인카드에서 구조적으로 `'_'`(EDA 3절)라 raw_payload 여부와 무관하게 신뢰 불가 | 사실상 가장 구하기 어려움 |

**사용 원칙**: `raw_payload` 스키마가 확정되기 전까지는 모델을 **Tier 0(+가용성 높은 Tier 1)만으로** 먼저 만들고,
Tier 2·3은 "확보되면 붙이는 보강 피처"로 취급한다. 아래 `select_features()`로 어떤 조합을 쓸지 명시적으로 고른다.


In [12]:
ID_COLS = ['카드KEY', '가맹점KEY', '승인SEQ']

# 이상거래유형(카드사 부정사용 세부유형 코드)·이상거래설명(자유텍스트, 96% 결측)은 데이터셋 자체에서 제거한다.
# 세부유형은 카드사 자체 분류 체계라 정산담당자 판단 기준과 무관하고(EDA 10-3절), 설명은 고카디널리티
# 자유텍스트라 활용도가 낮다. 남기는 라벨은 `이상거래여부` 하나뿐이다.
train_df = train_df.drop(columns=['이상거래유형', '이상거래설명'])
test_df = test_df.drop(columns=['이상거래유형', '이상거래설명'])

LABEL_COLS = ['이상거래여부']  # EDA 10-3절: 카드사 라벨이지 정산담당자 라벨이 아님 — 참고용/타깃 용도로만
RAW_DATE_COLS = ['기준년월', '승인일자']  # 파생 변수(거래일자 등)로 대체 가능

# --- Tier 0: transactions 핵심 필드(카드KEY/금액/시각)만으로 계산 가능 ---
# [변경] 가맹점 기준 파생 피처(가맹점평균금액_확장, 가맹점첫거래여부)는 5절에서 생성하지 않으므로 여기서도 제외.
TIER0_TRANSACTION_SAFE = [
    '승인시간대', '통합승인금액',
    '거래일자', '거래연월', '거래요일_한글', '시간대구간', '월말여부', '취소성거래_추정',
    '최근7일사용횟수', '카드누적사용액',
    '사용자평균사용액_확장', '사용자표준편차_확장', '거래금액_Zscore_확장', '카드첫거래여부',
]

# --- Tier 1: 카드/거래 정산 메타데이터. 법인카드 계정 관리(한도·구분·할부 등)에 필요해
#     카드사가 기본 제공할 가능성이 높은 필드들 — 별도 외부 enrichment 없이 raw_payload 파싱만으로 얻을 수 있을 것으로 추정.
TIER1_CARD_SETTLEMENT_META = [
    '개인법인구분코드_회원', '국내해외여부',
    '카드이용한도금액', '카드이용한도금액_구간', '법인카드_한도미기재추정',
    '승인거래코드', '승인발생경로코드', '로고구분코드',
    '일시불할부구분코드', '카드구분코드', '할부가능개월수',
]

# --- Tier 2: 사용자·부서·직급 등 조직 데이터. 이 데이터셋엔 없다 — users/teams 조인이 실제로 붙어야 채워짐 ---
TIER2_ORG_CONTEXT = []  # placeholder: 예) 부서, 직급, 팀 평균 대비 지출 배수 등 (EDA "추가 확보하면 좋은 데이터")

# --- Tier 3: 가맹점 속성 전체 + 카드홀더 인적정보 — 현실적으로 가장 구하기 어려움.
#     가맹점 속성: MCC는 카드사 제휴 post-MVP로 미뤄져 있고, 자체 카카오/웹검색 캐스케이드(§7-1)는
#     {industry_code, industry_label, confidence, source}라는 다른 스키마를 만들어서 이 원본 컬럼 값
#     자체는 어느 경로로도 재현되지 않는다. 전월_매출/개인법인구분코드_가맹점도 카드 거래 속성이 아니라
#     가맹점 속성이라 같은 묶음으로 옮겼다.
#     인적정보(연령/남녀구분코드): 법인카드는 이 필드가 구조적으로 '_'(EDA 3절)라 raw_payload 제공 여부와
#     무관하게 신뢰할 수 없다.
#     [참고] 이 Tier의 원본 컬럼들(가맹점광역시도코드 등)은 여전히 df에 남아있지만, 이를 바탕으로 한 파생
#     피처(가맹점평균금액_확장 등)는 5절에서 만들지 않는다 — 가맹점 신호는 RAG(case_history)가 담당.
TIER3_MERCHANT_AND_DEMOGRAPHIC = [
    '가맹점광역시도코드', '가맹점승인업종코드', '가맹점형태구분코드', '가맹점상태코드',
    '가맹점여부_신규', '가맹점누적매출금액_구간화', '인터넷판매여부',
    '개인법인구분코드_가맹점', '전월_매출건수', '전월_매출금액',
    '전월매출건수_음수여부', '전월매출금액_음수여부',
    '연령', '남녀구분코드',
    '경과일수_최종이용일자',  # EDA_v3 9-1절: 산정 기준 시점 자체가 미검증이라 잠정적으로 이 그룹에 둔다.
]

FEATURE_TIERS = {
    'tier0_transaction_safe': TIER0_TRANSACTION_SAFE,
    'tier1_card_settlement_meta': TIER1_CARD_SETTLEMENT_META,
    'tier2_org_context': TIER2_ORG_CONTEXT,
    'tier3_merchant_and_demographic': TIER3_MERCHANT_AND_DEMOGRAPHIC,
}

feature_cols = sum(FEATURE_TIERS.values(), [])

# 기존에 컬럼 기준(제외 목록 방식)으로 뽑았던 피처 집합과 정확히 같은지 검증 — 하나도 빠뜨리거나 중복하지 않았는지 확인
_legacy_feature_cols = [c for c in train_df.columns if c not in ID_COLS + LABEL_COLS + RAW_DATE_COLS + ['fold']]
assert set(feature_cols) == set(_legacy_feature_cols), (
    set(_legacy_feature_cols) ^ set(feature_cols)
)
assert len(feature_cols) == len(set(feature_cols)), "Tier 간 중복된 컬럼이 있습니다."

for tier_name, cols in FEATURE_TIERS.items():
    print(f"[{tier_name}] {len(cols)}개")
    print(cols)
    print()

print(f"전체 추천 피처: {len(feature_cols)}개 (Tier 0~3 합)")


[tier0_transaction_safe] 14개
['승인시간대', '통합승인금액', '거래일자', '거래연월', '거래요일_한글', '시간대구간', '월말여부', '취소성거래_추정', '최근7일사용횟수', '카드누적사용액', '사용자평균사용액_확장', '사용자표준편차_확장', '거래금액_Zscore_확장', '카드첫거래여부']

[tier1_card_settlement_meta] 11개
['개인법인구분코드_회원', '국내해외여부', '카드이용한도금액', '카드이용한도금액_구간', '법인카드_한도미기재추정', '승인거래코드', '승인발생경로코드', '로고구분코드', '일시불할부구분코드', '카드구분코드', '할부가능개월수']

[tier2_org_context] 0개
[]

[tier3_merchant_and_demographic] 15개
['가맹점광역시도코드', '가맹점승인업종코드', '가맹점형태구분코드', '가맹점상태코드', '가맹점여부_신규', '가맹점누적매출금액_구간화', '인터넷판매여부', '개인법인구분코드_가맹점', '전월_매출건수', '전월_매출금액', '전월매출건수_음수여부', '전월매출금액_음수여부', '연령', '남녀구분코드', '경과일수_최종이용일자']

전체 추천 피처: 40개 (Tier 0~3 합)


In [13]:
def select_features(tiers):
    '''사용할 Tier 이름들을 받아 피처 목록을 반환한다.

    raw_payload 스키마가 확정되기 전까지는 select_features(['tier0_transaction_safe'])처럼
    Tier 0(+ merchant_categories 연동이 끝났다면 tier1)만 쓰는 모델을 먼저 만드는 것을 권장한다.
    '''
    cols = []
    for t in tiers:
        cols += FEATURE_TIERS[t]
    return cols


prod_minimal_feats = select_features(['tier0_transaction_safe'])
prod_with_settlement_meta_feats = select_features(['tier0_transaction_safe', 'tier1_card_settlement_meta'])
benchmark_feats = select_features(list(FEATURE_TIERS.keys()))

print(f"프로덕션 최소(Tier 0)만: {len(prod_minimal_feats)}개")
print(f"Tier 0 + 카드/거래 정산 메타(Tier 1): {len(prod_with_settlement_meta_feats)}개")
print(f"AI Hub 벤치마크 전체(Tier 0~3): {len(benchmark_feats)}개")

프로덕션 최소(Tier 0)만: 14개
Tier 0 + 카드/거래 정산 메타(Tier 1): 25개
AI Hub 벤치마크 전체(Tier 0~3): 40개



## 9. Train 내부 검증 분할 (StratifiedGroupKFold, 카드 단위)

`test_df`는 최종 평가 전까지 손대지 않는다. 하이퍼파라미터 튜닝·조기종료·룰 임계값 선택(EDA 8절)을 위해
`train_df` 안에서 별도 검증 fold를 만든다.

- **카드(`카드KEY`) 단위로 그룹화**한다. EDA_v3 11절에서 카드가 분기를 넘어 평균 92.3% 재사용됨을 확인했고, 이 노트북의
  Expanding Window 피처(`사용자평균사용액_확장` 등)는 카드별 과거 이력을 담고 있다. 같은 카드의 거래가 fold-train과
  fold-valid에 걸쳐 있으면 "그 카드의 평소 패턴"이 양쪽에 새어 들어가 검증 성능이 낙관적으로 부풀 수 있으므로,
  한 카드의 모든 거래는 반드시 같은 fold에 속하게 한다.
- **`이상거래여부`로 층화(stratify)**한다. 클래스 불균형이 약 26:1이라 단순 GroupKFold만 쓰면 fold마다 이상거래
  비율이 들쭉날쭉할 수 있다. `StratifiedGroupKFold`는 그룹 제약을 지키면서 라벨 비율도 최대한 균등하게 맞춘다.


In [14]:
from sklearn.model_selection import StratifiedGroupKFold

N_SPLITS = 5
sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

train_df['fold'] = -1
for fold, (_, valid_idx) in enumerate(
    sgkf.split(train_df, train_df['이상거래여부'], groups=train_df['카드KEY'])
):
    train_df.loc[valid_idx, 'fold'] = fold

assert (train_df['fold'] >= 0).all(), "fold가 배정되지 않은 행이 있습니다."
print("fold별 행 수:")
print(train_df['fold'].value_counts().sort_index())

fold별 행 수:
fold
0    296613
1    296522
2    296582
3    296681
4    296571
Name: count, dtype: int64


In [15]:
# (1) 그룹 무결성: 같은 카드가 서로 다른 fold에 걸쳐 있으면 안 된다
card_fold_counts = train_df.groupby('카드KEY')['fold'].nunique()
n_leaked_cards = (card_fold_counts > 1).sum()
print(f"2개 이상 fold에 걸쳐 있는 카드 수: {n_leaked_cards} (0이어야 정상)")
assert n_leaked_cards == 0, "카드가 여러 fold에 걸쳐 있습니다 — 그룹 제약이 깨졌습니다."

# (2) 층화 품질: fold별 이상거래 비율이 전체 비율(3.687%)과 비슷한지 확인
fold_fraud_rate = train_df.groupby('fold')['이상거래여부'].mean() * 100
overall_rate = train_df['이상거래여부'].mean() * 100
print(f"\nfold별 이상거래 비율(%) (전체 대비 {overall_rate:.3f}%):")
print(fold_fraud_rate.round(3))

# 사용 예시: 특정 fold를 검증셋으로 쓰려면
# f = 0
# tr2 = train_df[train_df['fold'] != f]
# va = train_df[train_df['fold'] == f]

2개 이상 fold에 걸쳐 있는 카드 수: 0 (0이어야 정상)

fold별 이상거래 비율(%) (전체 대비 3.750%):
fold
0    3.754
1    3.740
2    3.748
3    3.763
4    3.746
Name: 이상거래여부, dtype: float64


## 10. 저장

In [16]:
# 이 환경에는 parquet 엔진(pyarrow/fastparquet)이 정상 설치되지 않아(컴파일된 확장 모듈 DLL 로드 실패) CSV로 저장한다.
# CSV는 dtype을 보존하지 않으므로 다시 읽을 때 아래를 재적용해야 한다(리뷰 통합본 경미 사항):
#   - category dtype 컬럼: 6절 CATEGORICAL_COLS 목록 기준으로 astype('category') 재적용
#   - `카드이용한도금액_구간`: CATEGORICAL_COLS에는 없고 4절에서 pd.Categorical로 별도 생성했으므로 함께 재캐스팅 필요
#   - `거래일자`: pd.to_datetime(df['거래일자'])로 재변환
#   - `거래연월`: pd.PeriodIndex(df['거래연월'], freq='M')로 재변환(또는 문자열 그대로 쓰고 필요할 때만 변환)
# 참고: test_processed.csv에만 `카드_train노출여부`(7-1절, 평가 진단용) 컬럼이 추가로 들어있다 —
#       train_df에는 없는 컬럼이라 두 파일의 컬럼 수가 다른 게 정상이다. 모델 학습 시 feature_cols에서
#       사용하지 않고, 평가 단계에서 세그먼트 분리용으로만 참조한다.
train_out = os.path.join(OUT_DIR, 'train_processed.csv')
test_out = os.path.join(OUT_DIR, 'test_processed.csv')

train_df.to_csv(train_out, index=False, encoding='utf-8')
test_df.to_csv(test_out, index=False, encoding='utf-8')

print(f"저장 완료: {train_out}  ({os.path.getsize(train_out)/1024/1024:.1f} MB)")
print(f"저장 완료: {test_out}  ({os.path.getsize(test_out)/1024/1024:.1f} MB)")

# 피처 Tier 정의도 함께 저장해, 이후 모델링 노트북이 이 전처리를 다시 실행하지 않고도
# select_features()와 동일한 정보를 로드해서 쓸 수 있게 한다.
tiers_out = os.path.join(OUT_DIR, 'feature_tiers.json')
with open(tiers_out, 'w', encoding='utf-8') as f:
    json.dump(FEATURE_TIERS, f, ensure_ascii=False, indent=2)
print(f"저장 완료: {tiers_out}")


저장 완료: ../.data/processed/train_processed.csv  (375.7 MB)
저장 완료: ../.data/processed/test_processed.csv  (121.1 MB)
저장 완료: ../.data/processed/feature_tiers.json


> **[가맹점 피처 제외 + 세그먼트 플래그 추가 반영 — 재실행 필요]** 이 노트북은 가맹점 기준 파생 피처
> (`가맹점평균금액_확장`, `가맹점첫거래여부`)를 5절에서 아예 생성하지 않으며, 7-1절에서 test 카드 세그먼트
> 플래그(`카드_train노출여부`)를 새로 추가한다. `train_processed.csv`/`test_processed.csv`를 다시
> 생성하려면 노트북을 처음부터 재실행해야 한다.
> 다시 생성하려면 노트북을 처음부터 재실행해야 한다.

## 요약

**최종 사용 피처 (40개, 8절 4-Tier 기준 — 프로덕션 스키마 대응력 순)**

| Tier | 개수 | 컬럼 | 프로덕션 가용성 |
|---|---|---|---|
| Tier 0 (transactions 핵심필드만으로 계산) | 14개 | 승인시간대, 통합승인금액, 거래일자\*, 거래연월\*, 거래요일_한글, 시간대구간, 월말여부, 취소성거래_추정, 최근7일사용횟수, 카드누적사용액, 사용자평균사용액_확장, 사용자표준편차_확장, 거래금액_Zscore_확장, 카드첫거래여부 | 거의 확실히 유지됨 |
| Tier 1 (카드/거래 정산 메타데이터) | 11개 | 개인법인구분코드_회원, 국내해외여부, 카드이용한도금액, 카드이용한도금액_구간, 법인카드_한도미기재추정, 승인거래코드, 승인발생경로코드, 로고구분코드, 일시불할부구분코드, 카드구분코드, 할부가능개월수 | 법인카드 계정 관리에 필요해 카드사가 기본 제공할 가능성 높음(§11.2 확정 전) |
| Tier 2 (조직 데이터) | 0개 | (placeholder — 부서/직급 등, EDA "추가 확보하면 좋은 데이터") | 이 데이터셋엔 없음, `users`/`teams` 조인 필요 |
| Tier 3 (가맹점 속성 + 인적정보) | 15개 | 가맹점광역시도코드, 가맹점승인업종코드, 가맹점형태구분코드, 가맹점상태코드, 가맹점여부_신규, 가맹점누적매출금액_구간화, 인터넷판매여부, 개인법인구분코드_가맹점, 전월_매출건수, 전월_매출금액, 전월매출건수_음수여부, 전월매출금액_음수여부, 연령, 남녀구분코드, 경과일수_최종이용일자 | 사실상 가장 구하기 어려움 — 가맹점 속성은 MCC가 post-MVP로 미뤄져 있고 자체 카카오/웹검색 캐스케이드는 다른 스키마를 만들어 원본 재현 불가. 인적정보는 법인카드에서 구조적으로 `'_'`(EDA 3절)라 무관하게 신뢰 불가 |

\* `거래일자`(datetime)·`거래연월`(Period)는 dtype이 그대로라 모델에 바로 넣으면 에러가 난다 — 모델링 단계에서 숫자로 변환하거나 제외해야 한다.

`select_features([...])`로 Tier 조합을 골라 쓸 수 있고, `FEATURE_TIERS`는 `.data/processed/feature_tiers.json`으로도 저장했다.

**데이터셋에서 완전히 삭제한 컬럼**
- `이상거래유형`, `이상거래설명` — 카드사 자체 부정사용 세부분류/자유텍스트로 정산담당자 판단 기준과 무관하고
  (EDA 10-3절), 활용도가 낮아 8절에서 `train_df`/`test_df`에서 완전히 제거했다.

**남아있지만 `feature_cols`에서는 제외한 컬럼**
- ID: `카드KEY`, `가맹점KEY`, `승인SEQ`
- 라벨: `이상거래여부` (EDA 10-3절: 카드사 라벨 — 타깃 후보로만 별도 취급, 그대로 피처화 금지)
- 원본 날짜 코드: `기준년월`, `승인일자` (파생 변수로 대체)
- `fold`: 9절에서 만든 검증용 컬럼, 피처 아님
- 가맹점 원본 속성 컬럼(가맹점광역시도코드 등, Tier 3): 컬럼 자체는 df에 남아있으나 파생 확장 피처는 생성하지 않음
- `카드_train노출여부` (test_df에만 존재, 7-1절): 평가 진단용 세그먼트 플래그, 학습 피처 아님

**이 노트북에서 한 일**
1. `train`/`test` 원본 파일이 같은 모집단의 무작위 분할임을 확인했다. 처음엔 그 경계를 유지한 채 시간 기반
   피처만 두 데이터를 합쳐서 계산했으나, 리뷰(🟠)에서 그것만으로는 test가 train 피처 계산을 오염시키는 문제가
   남는다는 지적을 받아 **원본 경계를 버리고 `거래일자` 기준(~2023 train / 2024 test)으로 재분할**했다(3-1절).
2. `'_'` 플레이스홀더를 명시적 NaN으로 정규화했다.
3. 날짜·시간대 파생 변수를 추가했다(거래일자/거래연월/거래요일/시간대구간/월말여부).
4. 0원 거래·전월 실적 음수·법인카드 한도 미기재를 **삭제 대신 플래그**로 남겼다.
5. Look-ahead 피처(EDA 9절 버전)는 만들지 않고, Expanding Window 기반 leak-free 피처(EDA 9-1절 버전, 카드 단위만)만 생성했다.
6. 콜드스타트(첫 거래) 상태를 `fillna`로 지우지 않고 플래그+결측으로 보존했다(EDA 10-4절 교훈).
7. 범주형 변수를 `category` dtype으로 정리하고(원-핫 인코딩 없음), 원본 컬럼은 삭제 없이 모두 보존했다.
8. `train_df`에 카드 단위 `StratifiedGroupKFold` 기반 `fold` 컬럼을 추가해, 모델 개발 시 test를 건드리지 않고도 검증할 수 있게 했다.
9. 피처를 프로덕션 스키마 기준 4-Tier(transactions 핵심필드 / 카드·거래 정산 메타데이터 / 조직 데이터 / 가맹점 속성+인적정보)로 재구성해, raw_payload 확정 전에도 `select_features()`로 안전한 조합만 골라 쓸 수 있게 했다.
10. `이상거래유형`·`이상거래설명`을 데이터셋에서 완전히 삭제했다(카드사 세부분류/자유텍스트, 정산담당자 판단 기준과 무관·활용도 낮음).
11. **Train/Test를 날짜 기준으로 재분할했다** — 원본 무작위 분할은 look-ahead는 막아도 "test가 train 피처
    계산에 개입하는 오염"은 막지 못한다는 지적(🟠 구조적 이슈)에 따라, 연도별 이상거래 비율 안정성(3.44~4.12%,
    3-1절)을 확인한 뒤 `거래일자 < 2024-01-01`을 기준으로 train(~2023)/test(2024)를 다시 나눴다. test 비중은
    원본(11.1%)보다 커졌다(약 24%).
12. **[변경] 가맹점 기준 파생 피처(`가맹점평균금액_확장`, `가맹점첫거래여부`)를 완전히 제외했다** — 카드 기준
    분할을 병행 검토하는 과정에서, 가맹점처럼 여러 카드가 공유하는 대상 기준 통계는 분할 방식과 무관하게 별도의
    시간축 정합성 이슈를 계속 만든다는 점을 확인했다. 가맹점 신호는 이 지도/비지도 피처 파이프라인이 아니라
    RAG(Chroma `case_history`)가 담당하도록 역할을 분리하기로 하고, 해당 피처 계산 코드를 5절에서 제거했다.
    (참고: 이전 버전에서 지적됐던 버그1은 이 피처가 없어짐에 따라 자연히 해소됐다.)

13. **[신규] test_df에 카드 재사용 세그먼트 플래그를 추가했다** — `카드_train노출여부`(7-1절) 컬럼으로
    test 카드를 "train에서 이미 본 카드(재사용)"와 "test 최초 등장 카드(신규)"로 구분해뒀다. 모델이
    카드 통계 피처(사용자평균사용액_확장 등)에 얼마나 의존하는지, 신규 카드 유입 시 실제 대응력이 어느
    정도인지를 평가 단계에서 세그먼트별로 나눠 확인하기 위한 진단용 컬럼이며, `feature_cols`에는 포함하지
    않는다.

**의도적으로 하지 않은 것**
- **스케일링**: EDA 10절에서 권장한 트리 기반 앙상블(LightGBM/XGBoost)은 스케일링이 필요 없다. 선형/거리 기반 모델을
  쓴다면 이 노트북 이후 단계에서 `feature_cols` 중 수치형만 골라 별도로 스케일링해야 한다.
- **결측치 임퓨테이션**: 트리 기반 모델은 결측을 네이티브로 처리할 수 있고, 임의로 채우면 콜드스타트 신호가 사라진다
  (EDA 10-4절). 결측을 반드시 채워야 하는 모델을 쓸 경우, 최소한 `카드첫거래여부` 플래그는 함께 넣어야 한다.
- **`이상거래여부`를 곧바로 지도학습 타깃으로 확정하는 것**: EDA 10-3절 결론대로 이 라벨은 카드사 부정사용 탐지
  기준이지 정산 담당자의 승인/반려 기준이 아니다. 이 노트북은 라벨을 보존만 하며, 실제 타깃 확정은 모델링 단계의
  판단으로 남겨둔다.
- **가맹점 기준 이상탐지 신호**: 위 12번 참고 — RAG 파이프라인으로 역할을 이관했으므로 이 지도/비지도 모델
  피처셋에는 포함하지 않는다.


---
# Part 2 — 피처셋 실험: Tier0 vs Tier0+1 (원본: `법인카드_이상거래_모델링_v1_Tier0vs1_비교.ipynb`)

> **주의**: 이 Part는 Part 1이 저장한 `train_processed.csv` / `feature_tiers.json`을 새로 로드해 `train_df`를
> **덮어쓴다**. 또한 여기서 정의하는 `build_model_matrix()`는 Part 3에서 최종 피처셋 전용 버전으로 다시 정의된다.


# 법인카드 이상거래 탐지 — Tier 0 vs Tier 0+1 비교 실험

**목적**: 1차 베이스라인(`모델링_v1/v2`)에서 Tier 0(14개)만으로 Isolation Forest를 돌렸을 때
PR-AUC 0.4711, recall@top5% 0.6300이 나왔다. 하이퍼파라미터 튜닝은 개선폭이 노이즈 수준(<1%)이라
포기했으므로(fold 표준편차 0.051 > 튜닝 개선폭 0.0012), 이번엔 **Tier 1(카드/거래 정산 메타데이터,
11개)을 추가했을 때 성능이 실제로 개선되는지**를 확인한다.

이 근거는 raw_payload 스키마 확정 논의(Open Issue #6-7)에도 쓰인다 — 개선폭이 크면 raw_payload에
카드 메타데이터를 반드시 포함시켜야 한다는 근거가 되고, 미미하면 Tier 0만으로도 충분하다는 근거가 된다.

**설계 원칙**
- 같은 노트북 안에서 Tier 0만 / Tier 0+1을 **동일한 fold, 동일한 하이퍼파라미터**로 평가해 공정하게 비교한다
  (베이스라인 노트북과 별도 실행이면 fold 셔플이나 환경 차이로 완벽히 같은 조건이라 보장하기 어려움)
- 하이퍼파라미터는 튜닝에서 개선 효과가 없었던 걸 확인했으므로 **베이스라인 값(n_estimators=200, max_samples='auto')을
  그대로 사용** — 지금 궁금한 건 "피처를 늘렸을 때 효과"이지 "하이퍼파라미터 효과"가 아니므로, 두 변수를 동시에
  바꾸면 어느 쪽 덕분에 성능이 변했는지 구분할 수 없다
- `test_df`는 이번에도 열지 않는다 (5단계 범위, fold 평가만)


## 1. 라이브러리 & 데이터 로드

In [1]:
import json
import os

import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score

pd.set_option('display.max_columns', 60)

DATA_DIR = '../.data/processed'
TRAIN_PATH = os.path.join(DATA_DIR, 'train_processed.csv')
TIERS_PATH = os.path.join(DATA_DIR, 'feature_tiers.json')

train_df = pd.read_csv(TRAIN_PATH, low_memory=False)
with open(TIERS_PATH, encoding='utf-8') as f:
    FEATURE_TIERS = json.load(f)

print(f"train_df: {train_df.shape}")
print(f"Tier 0: {len(FEATURE_TIERS['tier0_transaction_safe'])}개, "
      f"Tier 1: {len(FEATURE_TIERS['tier1_card_settlement_meta'])}개")


train_df: (1482969, 47)
Tier 0: 14개, Tier 1: 11개


## 2. dtype 재적용

Tier 0 재적용(요일/시간대구간 category, 거래일자 datetime, 월말여부 bool)에 더해, Tier 1에서 쓰는 컬럼 중
전처리 노트북 6절 `CATEGORICAL_COLS`에 있던 7개(`개인법인구분코드_회원`, `국내해외여부`, `승인거래코드`,
`승인발생경로코드`, `로고구분코드`, `일시불할부구분코드`, `카드구분코드`)를 category로, 4절에서 별도
`pd.Categorical`로 만들었던 `카드이용한도금액_구간`(순서형)을 다시 캐스팅한다.

In [2]:
# Tier 0
train_df['거래요일_한글'] = train_df['거래요일_한글'].astype('category')
train_df['시간대구간'] = train_df['시간대구간'].astype('category')
train_df['거래일자'] = pd.to_datetime(train_df['거래일자'])
train_df['월말여부'] = train_df['월말여부'].astype(bool)

# Tier 1 — 명목형(순서 없음)
TIER1_NOMINAL_CAT = [
    '개인법인구분코드_회원', '국내해외여부', '승인거래코드',
    '승인발생경로코드', '로고구분코드', '일시불할부구분코드', '카드구분코드',
]
for col in TIER1_NOMINAL_CAT:
    train_df[col] = train_df[col].astype('category')

# Tier 1 — 순서형 (전처리 4절: pd.Categorical(categories=[0, 1e6, 5e6, 1e7], ordered=True))
train_df['카드이용한도금액_구간'] = pd.Categorical(
    train_df['카드이용한도금액_구간'],
    categories=[0, 1_000_000, 5_000_000, 10_000_000],
    ordered=True,
)

print("Tier 1 범주형 컬럼별 고유값 개수 (원-핫 시 차원 폭발 여부 확인):")
for col in TIER1_NOMINAL_CAT:
    print(f"  {col}: {train_df[col].nunique(dropna=True)}개")


Tier 1 범주형 컬럼별 고유값 개수 (원-핫 시 차원 폭발 여부 확인):
  개인법인구분코드_회원: 2개
  국내해외여부: 2개
  승인거래코드: 5개
  승인발생경로코드: 4개
  로고구분코드: 7개
  일시불할부구분코드: 3개
  카드구분코드: 5개


## 3. 모델 입력 행렬 준비 함수 (Tier 0 / Tier 0+1 공용)

**[수정] 이전 실행에서 두 가지 문제가 확인되어 인코딩 방식을 바꾼다.**

1. **`dummy_na=True`를 무조건 켜뒀던 게 문제였다.** `거래요일_한글`·`시간대구간`처럼 애초에 NaN이 없는
   컬럼에도 "NaN 여부" 더미 컬럼이 만들어져, 정보가 전혀 없는 상수(전부 0) 컬럼이 섞여 들어갔다. Isolation
   Forest는 분기 피처를 **완전히 무작위로** 고르기 때문에, 이런 무의미한 컬럼이 하나라도 있으면 분기 기회가
   그쪽으로 낭비되어 실제 이상치 판별력이 떨어진다. → **실제로 NaN이 있는 컬럼에만** `dummy_na`를 켠다.
2. **Tier 1 범주형을 원-핫으로 풀었더니 컬럼이 21→62개로 3배 늘면서 오히려 성능이 60% 가까이 떨어졌다.**
   저카디널리티(2~7개)라도 원-핫으로 풀면 대부분 값이 2개뿐인 이진 컬럼이 되는데, 이진 컬럼은 어디서 분기하든
   결과가 거의 같아 판별력이 거의 없다. Isolation Forest가 무작위로 피처를 고르는 구조상, 이런 저정보 컬럼
   비중이 커질수록 진짜 유용한 연속형 피처(카드누적사용액, Zscore 등)가 뽑힐 확률이 희석된다.
   → **Tier 1 범주형은 원-핫 대신 순서 코드(`.cat.codes`)로 인코딩**해 컬럼 수를 늘리지 않는다. `거래요일_한글`·
   `시간대구간`(Tier 0)은 원래도 성능이 잘 나왔던 조합이라 그대로 원-핫을 유지한다.

**[참고] `할부가능개월수`**: 일시불 거래는 할부 개월 수 개념이 없어 NaN으로 기록된 것으로 보이므로, median이
아니라 **0(할부 없음)으로 채운다.**

In [3]:
NAN_FILL_COLS = ['사용자평균사용액_확장', '사용자표준편차_확장', '거래금액_Zscore_확장']
ZERO_FILL_COLS = ['할부가능개월수']  # 일시불 거래는 개념 자체가 없어 0(할부 없음)으로 채움 — median 부적절

# Tier 0에서 원래도 잘 작동했던 두 컬럼만 원-핫 유지. 나머지 범주형(Tier 1 포함)은 전부 순서 코드로 인코딩해
# 컬럼 수가 불필요하게 늘어나 Isolation Forest의 무작위 피처 선택이 희석되는 걸 방지한다.
TIER0_ONEHOT_COLS = ['거래요일_한글', '시간대구간']
TIER0_DROP_FROM_MODEL = ['거래일자', '거래연월']
ORDINAL_COLS = ['카드이용한도금액_구간']  # 순서형(구간 클수록 값도 큼) — codes가 순서 정보를 보존


def build_model_matrix(df, tier_cols, fill_values):
    '''주어진 피처 컬럼 목록(Tier 0 또는 Tier 0+1)을 Isolation Forest 입력용 수치 행렬로 변환.'''
    cols = [c for c in tier_cols if c not in TIER0_DROP_FROM_MODEL]
    X = df[cols].copy()

    # 1) 콜드스타트 확장 통계 NaN → median
    present_nan_cols = [c for c in NAN_FILL_COLS if c in X.columns]
    if present_nan_cols:
        X[present_nan_cols] = X[present_nan_cols].fillna(fill_values[present_nan_cols])

    # 1-1) 일시불이라 개념 자체가 없는 컬럼(할부가능개월수) → 0
    present_zero_cols = [c for c in ZERO_FILL_COLS if c in X.columns]
    if present_zero_cols:
        X[present_zero_cols] = X[present_zero_cols].fillna(0)

    # 2) bool 컬럼 → int
    for col in X.columns:
        if X[col].dtype == bool:
            X[col] = X[col].astype(int)

    # 3) 순서형 범주(카드이용한도금액_구간) → codes
    for col in ORDINAL_COLS:
        if col in X.columns:
            X[col] = X[col].cat.codes

    # 4) 나머지 category 컬럼: Tier0 지정 2개만 원-핫(실제 NaN 있을 때만 dummy_na), 그 외(Tier1 명목형)는 전부 codes
    remaining_cat_cols = [c for c in X.columns if str(X[c].dtype) == 'category']
    onehot_cols = [c for c in remaining_cat_cols if c in TIER0_ONEHOT_COLS]
    code_cols = [c for c in remaining_cat_cols if c not in TIER0_ONEHOT_COLS]

    for col in code_cols:
        X[col] = X[col].cat.codes

    for col in onehot_cols:
        has_na = X[col].isna().any()
        X = pd.get_dummies(X, columns=[col], dummy_na=has_na)

    # 5) 안전망: 예상 밖 NaN이 남아있으면 조용히 숨기지 않고 출력한 뒤 0으로 채운다
    remaining_na = X.isna().sum()
    remaining_na = remaining_na[remaining_na > 0]
    if len(remaining_na) > 0:
        print(f"  [경고] 명시적으로 처리되지 않은 NaN 컬럼 발견 — 0으로 채우고 계속 진행합니다:")
        print(remaining_na)
        X = X.fillna(0)

    return X


_fill_values_full = train_df[NAN_FILL_COLS].median()
print("NaN 대체 median 값:")
print(_fill_values_full)


NaN 대체 median 값:
사용자평균사용액_확장        27775.739501
사용자표준편차_확장        128056.821605
거래금액_Zscore_확장        -0.146994
dtype: float64


## 4. Tier 0 / Tier 0+1 두 피처셋으로 각각 5-fold 평가

베이스라인과 동일한 하이퍼파라미터(`n_estimators=200`, `max_samples='auto'`, `contamination='auto'`)로,
같은 fold 루프를 두 피처셋에 대해 각각 실행한다. 결과를 나중에 하나의 함수로 재사용할 수 있도록
`run_fold_cv()`로 묶어둔다.

In [4]:
N_SPLITS = 5
TOP_K_FRACTIONS = [0.01, 0.05, 0.10]


def run_fold_cv(tier_cols, label, n_estimators=200, max_samples='auto'):
    X_full = build_model_matrix(train_df, tier_cols, _fill_values_full)
    print(f"\n[{label}] 모델 입력 행렬 shape: {X_full.shape}")
    assert X_full.isna().sum().sum() == 0, f"[{label}] 결측이 남아있습니다."

    fold_rows = []
    for f in range(N_SPLITS):
        fold_train_idx = train_df.index[train_df['fold'] != f]
        fold_valid_idx = train_df.index[train_df['fold'] == f]

        X_tr = X_full.loc[fold_train_idx]
        X_va = X_full.loc[fold_valid_idx]
        y_va = train_df.loc[fold_valid_idx, '이상거래여부'].values

        iso = IsolationForest(
            n_estimators=n_estimators,
            max_samples=max_samples,
            contamination='auto',
            random_state=42,
            n_jobs=-1,
        )
        iso.fit(X_tr)
        anomaly_score = -iso.decision_function(X_va)

        pr_auc = average_precision_score(y_va, anomaly_score)
        row = {'fold': f, 'pr_auc': pr_auc}

        order = np.argsort(-anomaly_score)
        n_pos_total = y_va.sum()
        for frac in TOP_K_FRACTIONS:
            k = max(1, int(len(y_va) * frac))
            top_k_idx = order[:k]
            row[f'recall@top{int(frac*100)}%'] = y_va[top_k_idx].sum() / n_pos_total if n_pos_total > 0 else np.nan
            row[f'precision@top{int(frac*100)}%'] = y_va[top_k_idx].sum() / k

        fold_rows.append(row)
        print(f"  [{label} | fold {f}] PR-AUC={pr_auc:.4f}  recall@top5%={row['recall@top5%']:.3f}")

    return pd.DataFrame(fold_rows)


tier0_cols = FEATURE_TIERS['tier0_transaction_safe']
tier01_cols = FEATURE_TIERS['tier0_transaction_safe'] + FEATURE_TIERS['tier1_card_settlement_meta']

results_tier0 = run_fold_cv(tier0_cols, 'Tier0')
results_tier01 = run_fold_cv(tier01_cols, 'Tier0+1')



[Tier0] 모델 입력 행렬 shape: (1482969, 21)
  [Tier0 | fold 0] PR-AUC=0.4483  recall@top5%=0.606
  [Tier0 | fold 1] PR-AUC=0.4824  recall@top5%=0.620
  [Tier0 | fold 2] PR-AUC=0.5137  recall@top5%=0.644
  [Tier0 | fold 3] PR-AUC=0.5167  recall@top5%=0.649
  [Tier0 | fold 4] PR-AUC=0.3946  recall@top5%=0.632
  [경고] 명시적으로 처리되지 않은 NaN 컬럼 발견 — 0으로 채우고 계속 진행합니다:
카드이용한도금액    152
dtype: int64

[Tier0+1] 모델 입력 행렬 shape: (1482969, 32)
  [Tier0+1 | fold 0] PR-AUC=0.2005  recall@top5%=0.379
  [Tier0+1 | fold 1] PR-AUC=0.2200  recall@top5%=0.387
  [Tier0+1 | fold 2] PR-AUC=0.2066  recall@top5%=0.351
  [Tier0+1 | fold 3] PR-AUC=0.2274  recall@top5%=0.392
  [Tier0+1 | fold 4] PR-AUC=0.1962  recall@top5%=0.347


## 5. 결과 비교

In [5]:
summary_cols = ['pr_auc'] + [f'{m}@top{int(f*100)}%' for f in TOP_K_FRACTIONS for m in ['recall', 'precision']]

tier0_summary = results_tier0[summary_cols].mean().rename('Tier0 (14개)')
tier01_summary = results_tier01[summary_cols].mean().rename('Tier0+1 (25개)')

comparison = pd.concat([tier0_summary, tier01_summary], axis=1)
comparison['개선폭'] = comparison.iloc[:, 1] - comparison.iloc[:, 0]
comparison['개선율(%)'] = (comparison['개선폭'] / comparison.iloc[:, 0] * 100).round(2)

print("Tier0 fold별 표준편차 (노이즈 기준선):")
print(results_tier0[summary_cols].std().round(4))
print()
comparison.round(4)


Tier0 fold별 표준편차 (노이즈 기준선):
pr_auc              0.0510
recall@top1%        0.0227
precision@top1%     0.0854
recall@top5%        0.0178
precision@top5%     0.0138
recall@top10%       0.0110
precision@top10%    0.0043
dtype: float64



,Tier0 (14개),Tier0+1 (25개),개선폭,개선율(%)
pr_auc,0.4711,0.2101,-0.2610,-55.41
recall@top1%,0.1828,0.0859,-0.0969,-53.03
precision@top1%,0.6856,0.3220,-0.3635,-53.03
recall@top5%,0.6300,0.3713,-0.2587,-41.06
precision@top5%,0.4726,0.2785,-0.1940,-41.06
recall@top10%,0.7317,0.5465,-0.1852,-25.31
precision@top10%,0.2744,0.2049,-0.0694,-25.31


## 6. 중복 컬럼 제거 + Tier 1 개별 기여도 진단 (leave-one-in ablation)

**중복 제거**: `카드이용한도금액`(원본 금액)과 `카드이용한도금액_구간`(순서 코드)은 실제로 같은 정보를
두 번 담고 있다(전처리 4절 확인: 원본 값이 정확히 {0, 1백만, 5백만, 1천만} 4개뿐). 중복된 정보는 그 정보가
분기에 뽑힐 확률만 두 배로 늘릴 뿐 새로운 판별력을 주지 않으므로, **원본 금액은 빼고 구간 컬럼만 남긴다.**

**개별 기여도 진단**: Tier0+1 전체를 합쳤을 때 왜 성능이 크게 떨어지는지 확인하기 위해, Tier 1의 각 컬럼을
**하나씩만** Tier0에 추가해 성능 변화를 본다. 특정 컬럼 하나가 크게 깎아먹으면 그 컬럼이 범인이고, 전부
고르게 조금씩 깎인다면 Isolation Forest의 무작위 피처 선택 특성상 생기는 일반적인 희석 효과로 봐야 한다.

속도를 위해 이 진단은 fold 0~2(3개)만 사용한다.

In [6]:
TIER1_DEDUP = [c for c in FEATURE_TIERS['tier1_card_settlement_meta'] if c != '카드이용한도금액']
print(f"중복 제거 후 Tier 1: {len(TIER1_DEDUP)}개")
print(TIER1_DEDUP)

ABLATION_FOLDS = [0, 1, 2]
ablation_rows = []

# 기준선: Tier0만 (이미 4절에서 5-fold로 구했지만, 공정 비교를 위해 여기서도 동일 3-fold로 다시 잰다)
baseline_3fold = []
X_tier0 = build_model_matrix(train_df, tier0_cols, _fill_values_full)
for f in ABLATION_FOLDS:
    fold_train_idx = train_df.index[train_df['fold'] != f]
    fold_valid_idx = train_df.index[train_df['fold'] == f]
    iso = IsolationForest(n_estimators=200, max_samples='auto', contamination='auto', random_state=42, n_jobs=-1)
    iso.fit(X_tier0.loc[fold_train_idx])
    score = -iso.decision_function(X_tier0.loc[fold_valid_idx])
    y_va = train_df.loc[fold_valid_idx, '이상거래여부'].values
    baseline_3fold.append(average_precision_score(y_va, score))

baseline_pr_auc_3fold = float(np.mean(baseline_3fold))
print(f"\nTier0 단독 (3-fold 기준) PR-AUC: {baseline_pr_auc_3fold:.4f}\n")

for col in TIER1_DEDUP:
    trial_cols = tier0_cols + [col]
    X_trial = build_model_matrix(train_df, trial_cols, _fill_values_full)

    pr_aucs = []
    for f in ABLATION_FOLDS:
        fold_train_idx = train_df.index[train_df['fold'] != f]
        fold_valid_idx = train_df.index[train_df['fold'] == f]
        iso = IsolationForest(n_estimators=200, max_samples='auto', contamination='auto', random_state=42, n_jobs=-1)
        iso.fit(X_trial.loc[fold_train_idx])
        score = -iso.decision_function(X_trial.loc[fold_valid_idx])
        y_va = train_df.loc[fold_valid_idx, '이상거래여부'].values
        pr_aucs.append(average_precision_score(y_va, score))

    pr_auc_mean = float(np.mean(pr_aucs))
    delta = pr_auc_mean - baseline_pr_auc_3fold
    ablation_rows.append({
        'tier1_컬럼': col,
        'pr_auc_mean_3fold': pr_auc_mean,
        'Tier0 대비 변화': delta,
        '변화율(%)': round(delta / baseline_pr_auc_3fold * 100, 2),
    })
    print(f"  Tier0 + {col:<15} → PR-AUC={pr_auc_mean:.4f}  (변화 {delta:+.4f}, {delta/baseline_pr_auc_3fold*100:+.1f}%)")

ablation_df = pd.DataFrame(ablation_rows).sort_values('pr_auc_mean_3fold', ascending=False).reset_index(drop=True)
ablation_df


중복 제거 후 Tier 1: 10개
['개인법인구분코드_회원', '국내해외여부', '카드이용한도금액_구간', '법인카드_한도미기재추정', '승인거래코드', '승인발생경로코드', '로고구분코드', '일시불할부구분코드', '카드구분코드', '할부가능개월수']

Tier0 단독 (3-fold 기준) PR-AUC: 0.4815

  Tier0 + 개인법인구분코드_회원     → PR-AUC=0.4248  (변화 -0.0567, -11.8%)
  Tier0 + 국내해외여부          → PR-AUC=0.4594  (변화 -0.0221, -4.6%)
  Tier0 + 카드이용한도금액_구간     → PR-AUC=0.4538  (변화 -0.0277, -5.7%)
  Tier0 + 법인카드_한도미기재추정    → PR-AUC=0.4250  (변화 -0.0565, -11.7%)
  Tier0 + 승인거래코드          → PR-AUC=0.4643  (변화 -0.0172, -3.6%)
  Tier0 + 승인발생경로코드        → PR-AUC=0.4282  (변화 -0.0533, -11.1%)
  Tier0 + 로고구분코드          → PR-AUC=0.4637  (변화 -0.0178, -3.7%)
  Tier0 + 일시불할부구분코드       → PR-AUC=0.5160  (변화 +0.0345, +7.2%)
  Tier0 + 카드구분코드          → PR-AUC=0.4255  (변화 -0.0560, -11.6%)
  Tier0 + 할부가능개월수         → PR-AUC=0.4412  (변화 -0.0403, -8.4%)


,tier1_컬럼,pr_auc_mean_3fold,Tier0 대비 변화,변화율(%)
0,일시불할부구분코드,0.515978,0.034485,7.16
1,승인거래코드,0.464286,-0.017207,-3.57
2,로고구분코드,0.463698,-0.017795,-3.70
3,국내해외여부,0.459440,-0.022054,-4.58
4,카드이용한도금액_구간,0.453836,-0.027657,-5.74
5,할부가능개월수,0.441203,-0.040290,-8.37
6,승인발생경로코드,0.428218,-0.053276,-11.06
7,카드구분코드,0.425511,-0.055982,-11.63
8,법인카드_한도미기재추정,0.424993,-0.056501,-11.73
9,개인법인구분코드_회원,0.424811,-0.056683,-11.77


## 7. 진단 결과를 바탕으로 최종 Tier0+1 피처셋 확정

6절 결과에서 **Tier0 단독보다 확실히 낮게 나온 컬럼**은 제외 후보로 표시하고, 나머지(중립~긍정 기여)만
모아 최종 Tier0+1 조합을 다시 구성한다. "확실히 낮다"의 기준은 3-fold 노이즈 폭(하이퍼파라미터 탐색 때
관찰된 표준편차 0.02~0.04 수준)보다 뚜렷하게 큰 하락으로 잡는다.

In [7]:
DROP_THRESHOLD = -0.02  # Tier0 대비 이보다 더 크게 떨어지면 제외 후보 (3-fold 노이즈 폭 감안)

drop_candidates = ablation_df[ablation_df['Tier0 대비 변화'] < DROP_THRESHOLD]['tier1_컬럼'].tolist()
keep_candidates = [c for c in TIER1_DEDUP if c not in drop_candidates]

print(f"제외 후보 ({len(drop_candidates)}개): {drop_candidates}")
print(f"유지 후보 ({len(keep_candidates)}개): {keep_candidates}")

final_tier01_cols = tier0_cols + keep_candidates
X_final = build_model_matrix(train_df, final_tier01_cols, _fill_values_full)
print(f"\n최종 Tier0+1(정제) 입력 행렬 shape: {X_final.shape}")

final_fold_rows = []
for f in range(N_SPLITS):
    fold_train_idx = train_df.index[train_df['fold'] != f]
    fold_valid_idx = train_df.index[train_df['fold'] == f]
    X_tr = X_final.loc[fold_train_idx]
    X_va = X_final.loc[fold_valid_idx]
    y_va = train_df.loc[fold_valid_idx, '이상거래여부'].values

    iso = IsolationForest(n_estimators=200, max_samples='auto', contamination='auto', random_state=42, n_jobs=-1)
    iso.fit(X_tr)
    anomaly_score = -iso.decision_function(X_va)

    pr_auc = average_precision_score(y_va, anomaly_score)
    row = {'fold': f, 'pr_auc': pr_auc}
    order = np.argsort(-anomaly_score)
    n_pos_total = y_va.sum()
    for frac in TOP_K_FRACTIONS:
        k = max(1, int(len(y_va) * frac))
        top_k_idx = order[:k]
        row[f'recall@top{int(frac*100)}%'] = y_va[top_k_idx].sum() / n_pos_total if n_pos_total > 0 else np.nan
        row[f'precision@top{int(frac*100)}%'] = y_va[top_k_idx].sum() / k
    final_fold_rows.append(row)
    print(f"[fold {f}] PR-AUC={pr_auc:.4f}  recall@top5%={row['recall@top5%']:.3f}")

results_tier01_refined = pd.DataFrame(final_fold_rows)


제외 후보 (7개): ['국내해외여부', '카드이용한도금액_구간', '할부가능개월수', '승인발생경로코드', '카드구분코드', '법인카드_한도미기재추정', '개인법인구분코드_회원']
유지 후보 (3개): ['승인거래코드', '로고구분코드', '일시불할부구분코드']

최종 Tier0+1(정제) 입력 행렬 shape: (1482969, 24)
[fold 0] PR-AUC=0.4684  recall@top5%=0.577
[fold 1] PR-AUC=0.4932  recall@top5%=0.595
[fold 2] PR-AUC=0.5382  recall@top5%=0.633
[fold 3] PR-AUC=0.5338  recall@top5%=0.628
[fold 4] PR-AUC=0.3730  recall@top5%=0.568


## 8. 최종 비교 — Tier0 vs Tier0+1(정제 전) vs Tier0+1(정제 후)

In [8]:
tier0_summary_final = results_tier0[summary_cols].mean().rename('Tier0 (14개)')
tier01_raw_summary = results_tier01[summary_cols].mean().rename('Tier0+1 정제 전')
tier01_refined_summary = results_tier01_refined[summary_cols].mean().rename('Tier0+1 정제 후')

final_comparison = pd.concat([tier0_summary_final, tier01_raw_summary, tier01_refined_summary], axis=1)
final_comparison.round(4)


,Tier0 (14개),Tier0+1 정제 전,Tier0+1 정제 후
pr_auc,0.4711,0.2101,0.4813
recall@top1%,0.1828,0.0859,0.1936
precision@top1%,0.6856,0.3220,0.7264
recall@top5%,0.6300,0.3713,0.6002
precision@top5%,0.4726,0.2785,0.4502
recall@top10%,0.7317,0.5465,0.7321
precision@top10%,0.2744,0.2049,0.2746


## 9. `일시불할부구분코드` 단독 검증 (5-fold 전체)

6절 ablation(3-fold)에서 Tier 1 10개 중 유일하게 양의 신호(+7.2%)를 보인 컬럼이다. 3-fold 노이즈였는지
확인하기 위해, **Tier0 + 이 컬럼 1개**만으로 4절과 동일한 5-fold 전체 평가를 다시 돌린다. 나머지 9개
Tier1 컬럼은 개별로도 마이너스였으므로 이 검증에서 제외한다.

In [9]:
results_tier0_installment = run_fold_cv(
    tier0_cols + ['일시불할부구분코드'], 'Tier0+일시불할부구분코드'
)



[Tier0+일시불할부구분코드] 모델 입력 행렬 shape: (1482969, 22)
  [Tier0+일시불할부구분코드 | fold 0] PR-AUC=0.4705  recall@top5%=0.586
  [Tier0+일시불할부구분코드 | fold 1] PR-AUC=0.5082  recall@top5%=0.610
  [Tier0+일시불할부구분코드 | fold 2] PR-AUC=0.5692  recall@top5%=0.658
  [Tier0+일시불할부구분코드 | fold 3] PR-AUC=0.5336  recall@top5%=0.631
  [Tier0+일시불할부구분코드 | fold 4] PR-AUC=0.3914  recall@top5%=0.593


## 10. 최종 판단 — Tier0 vs Tier0+일시불할부구분코드

In [10]:
tier0_final = results_tier0[summary_cols].mean().rename('Tier0 (14개)')
tier0_installment_final = results_tier0_installment[summary_cols].mean().rename('Tier0+일시불할부구분코드 (15개)')

installment_comparison = pd.concat([tier0_final, tier0_installment_final], axis=1)
installment_comparison['개선폭'] = installment_comparison.iloc[:, 1] - installment_comparison.iloc[:, 0]
installment_comparison['개선율(%)'] = (
    installment_comparison['개선폭'] / installment_comparison.iloc[:, 0] * 100
).round(2)

print("Tier0 fold별 표준편차 (노이즈 기준선, 참고용):")
print(results_tier0[summary_cols].std().round(4))
print()
installment_comparison.round(4)


Tier0 fold별 표준편차 (노이즈 기준선, 참고용):
pr_auc              0.0510
recall@top1%        0.0227
precision@top1%     0.0854
recall@top5%        0.0178
precision@top5%     0.0138
recall@top10%       0.0110
precision@top10%    0.0043
dtype: float64



,Tier0 (14개),Tier0+일시불할부구분코드 (15개),개선폭,개선율(%)
pr_auc,0.4711,0.4946,0.0235,4.98
recall@top1%,0.1828,0.1948,0.0121,6.60
precision@top1%,0.6856,0.7308,0.0453,6.60
recall@top5%,0.6300,0.6154,-0.0147,-2.33
precision@top5%,0.4726,0.4616,-0.0110,-2.33
recall@top10%,0.7317,0.7508,0.0192,2.62
precision@top10%,0.2744,0.2816,0.0072,2.62


## 요약

**비교 방법**: 동일한 fold(카드 단위 5-fold), 동일한 하이퍼파라미터(n_estimators=200, max_samples='auto')로
Tier 0(14개)와 Tier 0+1(25개)을 각각 5회씩 학습·평가해 PR-AUC·top-k recall/precision을 비교했다.

**판단 기준**: 5절 출력의 `개선율(%)`이 Tier0의 fold 표준편차(노이즈 수준, 대략 PR-AUC 기준 ±10% 내외)보다
뚜렷하게 크면 "Tier 1 추가가 실질적으로 유의미하다"고 볼 수 있고, 그 이하면(하이퍼파라미터 튜닝 때처럼)
노이즈 범위 안이라 raw_payload에 Tier 1을 반드시 포함시켜야 한다는 강한 근거로 쓰기 어렵다.

**9~10절: `일시불할부구분코드` 단독 검증**
- 6절 3-fold ablation에서 유일하게 양의 신호(+7.2%)를 보인 컬럼을 5-fold 전체로 재검증했다. 노이즈였는지,
  실제로 유의미한 개선인지는 10절 `installment_comparison`과 Tier0 fold 표준편차를 비교해 판단한다.

**6~8절 진단 결과 반영**
- `카드이용한도금액`(원본)과 `카드이용한도금액_구간`(순서 코드)이 완전히 중복된 정보라 원본은 제외했다.
- Tier 1 10개 컬럼을 하나씩 Tier0에 추가해봄으로써, 어느 컬럼이 성능을 깎아먹는지(또는 기여하는지) 개별
  확인했다(6절 `ablation_df`). 뚜렷하게 하락시키는 컬럼은 제외하고 나머지만 모아 최종 Tier0+1(정제) 조합을
  다시 구성해 5-fold로 재평가했다(7~8절).

**다음 결정**
- 개선폭이 크다 → raw_payload 스키마 논의(Open Issue #6-7)에 "카드 메타데이터 포함 필요"라는 정량적 근거로 제시
- 개선폭이 작다 → Tier 0만으로 MVP를 시작하고, raw_payload 확정 후 Tier 1을 재평가하는 쪽으로 결정
- 어느 쪽이든 이번 결과를 팀 문서에 남겨, 나중에 "왜 이 피처셋을 선택했는지" 근거로 활용


---
# Part 3 — 최종 학습 및 test 평가 (원본: `법인카드_이상거래_모델링_v2_최종test평가.ipynb`)

> **주의**: 이 Part는 `train_processed.csv`·`test_processed.csv`를 새로 로드하고, `build_model_matrix()`를
> 최종 피처셋(Tier0 + `일시불할부구분코드`) 전용으로 재정의한다. `test_df`는 이 Part에서 **딱 한 번만** 채점한다 —
> 결과를 보고 피처/하이퍼파라미터를 바꿔 재실행하면 그 자체가 test를 이용한 누수가 된다.


# 법인카드 이상거래 탐지 — 7단계: 최종 학습 및 test 평가 (딱 한 번)

**목적**: 5단계(fold 비교)에서 확정한 조합을 `train_df` 전체로 최종 학습하고, 지금까지 한 번도 열지 않았던
`test_df`(2024년)에 **이 노트북에서 딱 한 번만** 적용해 최종 성능을 확인한다.

**확정된 조합 (5단계 결과)**
- 알고리즘: Isolation Forest
- 피처셋: Tier0(14개) + `일시불할부구분코드`(1개) = 15개
- 하이퍼파라미터: `n_estimators=200`, `max_samples='auto'`, `contamination='auto'`
- fold 평균 성능(참고용, train 내부): PR-AUC 0.4946, recall@top5% 0.6154, recall@top10% 0.7508

**⚠️ 이 노트북의 규칙**
- `test_df`는 **이 노트북에서 딱 한 번만 채점**한다. 결과가 마음에 안 든다고 피처나 하이퍼파라미터를 바꿔서
  이 노트북을 반복 실행하면, 그 자체가 test를 이용한 새로운 형태의 과적합(누수)이 된다.
- 결과가 fold 평균과 크게 다르면(과적합/과소적합 정황), 원인 분석은 하되 **test 점수를 보고 하이퍼파라미터를
  다시 튜닝하지 않는다** — 그건 5단계로 돌아가 train 내부 fold로만 다시 실험해야 한다.
- 전체 성능과 함께, 전처리 7-1절에서 만든 `카드_train노출여부`(재사용 카드 vs 신규 카드) 세그먼트별 성능도
  반드시 같이 리포트한다.

## 1. 라이브러리 & 데이터 로드

In [1]:
import json
import os

import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score

pd.set_option('display.max_columns', 60)

DATA_DIR = '../.data/processed'
TRAIN_PATH = os.path.join(DATA_DIR, 'train_processed.csv')
TEST_PATH = os.path.join(DATA_DIR, 'test_processed.csv')
TIERS_PATH = os.path.join(DATA_DIR, 'feature_tiers.json')

train_df = pd.read_csv(TRAIN_PATH, low_memory=False)
test_df = pd.read_csv(TEST_PATH, low_memory=False)
with open(TIERS_PATH, encoding='utf-8') as f:
    FEATURE_TIERS = json.load(f)

print(f"train_df: {train_df.shape}")
print(f"test_df : {test_df.shape}")
print(f"test_df에 카드_train노출여부 존재: {'카드_train노출여부' in test_df.columns}")


train_df: (1482969, 47)
test_df : (469902, 47)
test_df에 카드_train노출여부 존재: True


## 2. dtype 재적용 (train, test 둘 다)

전처리 노트북 10절 안내대로, CSV에서 다시 읽으면 category/datetime dtype이 풀려있다. 최종 피처셋에 필요한
컬럼만 재적용한다: Tier0의 `거래요일_한글`·`시간대구간`(category), `거래일자`(datetime), `월말여부`(bool),
그리고 `일시불할부구분코드`(category).

In [2]:
for df in (train_df, test_df):
    df['거래요일_한글'] = df['거래요일_한글'].astype('category')
    df['시간대구간'] = df['시간대구간'].astype('category')
    df['거래일자'] = pd.to_datetime(df['거래일자'])
    df['월말여부'] = df['월말여부'].astype(bool)
    df['일시불할부구분코드'] = df['일시불할부구분코드'].astype('category')

test_df['카드_train노출여부'] = test_df['카드_train노출여부'].astype(bool)

print("재적용 후 dtype 확인:")
print(train_df[['거래요일_한글', '시간대구간', '거래일자', '월말여부', '일시불할부구분코드']].dtypes)


재적용 후 dtype 확인:
거래요일_한글            category
시간대구간              category
거래일자         datetime64[us]
월말여부                   bool
일시불할부구분코드          category
dtype: object


## 3. 모델 입력 행렬 준비 (최종 피처셋: Tier0 + 일시불할부구분코드)

5단계에서 검증한 방식을 그대로 따른다 — 확장 통계 NaN은 median(train 기준값을 test에도 동일하게 적용,
누수 방지를 위해 test 자체의 median을 다시 구하지 않음), `할부가능개월수`는 이번 최종 피처셋에 없으므로
해당 처리는 불필요, `일시불할부구분코드`는 저카디널리티라 원-핫 인코딩한다.

**⚠️ 피처 개수 표기 정정**: `FEATURE_TIERS['tier0_transaction_safe']`(14개) + `일시불할부구분코드`(1개) = 15개를
"최종 확정 피처셋"이라 불러왔지만, 이 중 `거래일자`·`거래연월`은 `DROP_FROM_MODEL`로 모델 입력에서 제외된다
(시간 식별자는 파생 피처 계산용이지 IsolationForest 입력이 아님). 따라서 **실제 모델에 들어가는 원본 컬럼은
13개**이며, 원-핫 인코딩 후 24개 컬럼이 된다. 팀 문서·발표 자료에는 "확정 피처셋 15개(그중 모델 직접 입력
13개, 2개는 파생 계산용 식별자)"로 명시할 것 — "15개로 학습했다"고만 쓰면 노트북과 불일치한다.


In [3]:
NAN_FILL_COLS = ['사용자평균사용액_확장', '사용자표준편차_확장', '거래금액_Zscore_확장']
ONEHOT_COLS = ['거래요일_한글', '시간대구간', '일시불할부구분코드']
DROP_FROM_MODEL = ['거래일자', '거래연월']

FINAL_FEATURE_COLS = FEATURE_TIERS['tier0_transaction_safe'] + ['일시불할부구분코드']
print(f"최종 피처 목록 ({len(FINAL_FEATURE_COLS)}개):")
print(FINAL_FEATURE_COLS)

# median은 반드시 train 기준으로만 계산 — test 정보가 전처리에 섞이지 않도록
_fill_values = train_df[NAN_FILL_COLS].median()
print("\nNaN 대체 median 값 (train 기준):")
print(_fill_values)


def build_model_matrix(df, fill_values=_fill_values):
    cols = [c for c in FINAL_FEATURE_COLS if c not in DROP_FROM_MODEL]
    X = df[cols].copy()
    X[NAN_FILL_COLS] = X[NAN_FILL_COLS].fillna(fill_values)
    X['월말여부'] = X['월말여부'].astype(int)
    X = pd.get_dummies(X, columns=ONEHOT_COLS, drop_first=False)

    remaining_na = X.isna().sum()
    remaining_na = remaining_na[remaining_na > 0]
    if len(remaining_na) > 0:
        print(f"  [경고] 처리 안 된 NaN 발견 — 0으로 채우고 진행:")
        print(remaining_na)
        X = X.fillna(0)
    return X


X_train = build_model_matrix(train_df)
X_test = build_model_matrix(test_df)

# train/test 컬럼이 원-핫 인코딩 후 서로 다를 수 있으므로(예: 특정 카테고리가 한쪽에만 존재) 맞춰준다
X_train, X_test = X_train.align(X_test, join='outer', axis=1, fill_value=0)

print(f"\nX_train shape: {X_train.shape}")
print(f"X_test  shape: {X_test.shape}")
assert X_train.isna().sum().sum() == 0
assert X_test.isna().sum().sum() == 0


최종 피처 목록 (15개):
['승인시간대', '통합승인금액', '거래일자', '거래연월', '거래요일_한글', '시간대구간', '월말여부', '취소성거래_추정', '최근7일사용횟수', '카드누적사용액', '사용자평균사용액_확장', '사용자표준편차_확장', '거래금액_Zscore_확장', '카드첫거래여부', '일시불할부구분코드']

NaN 대체 median 값 (train 기준):
사용자평균사용액_확장        27775.739501
사용자표준편차_확장        128056.821605
거래금액_Zscore_확장        -0.146994
dtype: float64

X_train shape: (1482969, 24)
X_test  shape: (469902, 24)


## 4. 최종 학습 (train 전체) 및 test 채점 (딱 한 번)

5단계에서 확정한 하이퍼파라미터 그대로, 이번엔 fold로 나누지 않고 **train_df 전체**로 학습한다.

In [4]:
final_model = IsolationForest(
    n_estimators=200,
    max_samples='auto',
    contamination='auto',
    random_state=42,
    n_jobs=-1,
)
final_model.fit(X_train)

test_anomaly_score = -final_model.decision_function(X_test)
test_df['anomaly_score'] = test_anomaly_score

print("test_df 채점 완료.")
print(f"anomaly_score 범위: {test_anomaly_score.min():.4f} ~ {test_anomaly_score.max():.4f}")


test_df 채점 완료.
anomaly_score 범위: -0.1226 ~ 0.2268


## 5. 전체 test 성능

In [5]:
TOP_K_FRACTIONS = [0.01, 0.03, 0.05, 0.10]

y_test = test_df['이상거래여부'].values
overall_pr_auc = average_precision_score(y_test, test_anomaly_score)

order = np.argsort(-test_anomaly_score)
n_pos_total = y_test.sum()

overall_row = {'pr_auc': overall_pr_auc, 'n': len(y_test), 'n_positive': int(n_pos_total)}
for frac in TOP_K_FRACTIONS:
    k = max(1, int(len(y_test) * frac))
    top_k_idx = order[:k]
    overall_row[f'recall@top{int(frac*100)}%'] = y_test[top_k_idx].sum() / n_pos_total
    overall_row[f'precision@top{int(frac*100)}%'] = y_test[top_k_idx].sum() / k

overall_summary = pd.Series(overall_row)
print("=== 전체 test 성능 ===")
print(overall_summary)

print(f"\n(참고) train 내부 5-fold 평균 PR-AUC: 0.4946 — 이 값과 비교해 과적합/과소적합 여부를 가늠할 수 있다.")


=== 전체 test 성능 ===
pr_auc                   0.586508
n                   469902.000000
n_positive           16394.000000
recall@top1%             0.248689
precision@top1%          0.867631
recall@top3%             0.525802
precision@top3%          0.611478
recall@top5%             0.663779
precision@top5%          0.463162
recall@top10%            0.790350
precision@top10%         0.275740
dtype: float64

(참고) train 내부 5-fold 평균 PR-AUC: 0.4946 — 이 값과 비교해 과적합/과소적합 여부를 가늠할 수 있다.


## 6. 세그먼트별 test 성능 — 재사용 카드 vs 신규 카드

전처리 7-1절에서 만든 `카드_train노출여부`로 나눠서, 모델이 카드 이력(재사용 카드)에 얼마나 의존하는지,
신규 카드(콜드스타트)에서는 실제로 얼마나 잘 작동하는지 확인한다.

In [6]:
def compute_metrics(y_true, scores, fracs=TOP_K_FRACTIONS):
    row = {'n': len(y_true), 'n_positive': int(y_true.sum())}
    if y_true.sum() == 0:
        row['pr_auc'] = np.nan
    else:
        row['pr_auc'] = average_precision_score(y_true, scores)
    order = np.argsort(-scores)
    n_pos = y_true.sum()
    for frac in fracs:
        k = max(1, int(len(y_true) * frac))
        top_k_idx = order[:k]
        row[f'recall@top{int(frac*100)}%'] = y_true[top_k_idx].sum() / n_pos if n_pos > 0 else np.nan
        row[f'precision@top{int(frac*100)}%'] = y_true[top_k_idx].sum() / k
    return row


segment_rows = {}
for seg_value, seg_label in [(True, '재사용 카드'), (False, '신규 카드')]:
    mask = test_df['카드_train노출여부'] == seg_value
    y_seg = test_df.loc[mask, '이상거래여부'].values
    score_seg = test_anomaly_score[mask.values]
    segment_rows[seg_label] = compute_metrics(y_seg, score_seg)

segment_summary = pd.DataFrame(segment_rows)
print("=== 세그먼트별 test 성능 ===")
segment_summary.round(4)


=== 세그먼트별 test 성능 ===


,재사용 카드,신규 카드
n,465328.0000,4574.0000
n_positive,16231.0000,163.0000
pr_auc,0.5865,0.6367
recall@top1%,0.2487,0.2331
precision@top1%,0.8674,0.8444
recall@top3%,0.5262,0.5767
precision@top3%,0.6118,0.6861
recall@top5%,0.6632,0.7423
precision@top5%,0.4627,0.5307
recall@top10%,0.7901,0.8282


## 7. 우선순위 랭킹 규칙 확정 (top3% 컷오프)

회계팀 검토 용량을 감안해 **상위 3%를 위험 후보로 분류**하기로 확정했다. 두 가지를 구분해서 확인한다.

- **(a) test 자체에서 정확히 상위 3%**: 5~6절 `TOP_K_FRACTIONS`에 이미 0.03을 추가해뒀으므로 그 결과를 그대로 참고
- **(b) 운영 시나리오 재현**: 실제 배포에서는 매 거래가 들어올 때마다 "미리 정해둔 고정 점수 임계값"과 비교해야
  하므로, **train 점수 분포의 97번째 백분위수를 임계값으로 고정**한 뒤 그걸 test에 적용했을 때 실제로 몇 %가
  걸리는지, 그 성능이 (a)와 얼마나 차이 나는지 확인한다. 차이가 크면 임계값이 시간에 따라 불안정하다는 신호다.

**⚠️ 문서화 보완**: (a) "이상적 top3%" 수치(recall 0.526 / precision 0.611)만 팀 문서에 남기면 실제 운영
성능으로 오해될 수 있다. 아래 (b) 고정 임계값 적용 결과(recall/precision)가 곧 운영 시 기대 성능이므로,
팀 공유 문서에는 (a)와 (b) 두 수치를 반드시 함께 기재한다.


In [7]:
train_anomaly_score = -final_model.decision_function(X_train)
THRESHOLD_PERCENTILE = 97  # 상위 3%
fixed_threshold = np.percentile(train_anomaly_score, THRESHOLD_PERCENTILE)
print(f"train 기준 고정 임계값(상위 3% 지점): {fixed_threshold:.4f}")

# (a) test 내 정확히 상위 3% (5절 결과에서 재확인)
k_a = max(1, int(len(y_test) * 0.03))
order_a = np.argsort(-test_anomaly_score)[:k_a]
recall_a = y_test[order_a].sum() / y_test.sum()
precision_a = y_test[order_a].sum() / k_a

# (b) train에서 정한 고정 임계값을 test에 그대로 적용
flagged_b = test_anomaly_score >= fixed_threshold
actual_pct_b = flagged_b.mean() * 100
recall_b = y_test[flagged_b].sum() / y_test.sum()
precision_b = y_test[flagged_b].sum() / flagged_b.sum() if flagged_b.sum() > 0 else np.nan

print(f"\n(a) test 내 정확히 top3% ({k_a:,}건): recall={recall_a:.3f}, precision={precision_a:.3f}")
print(f"(b) train 기준 고정 임계값 적용 (실제 {actual_pct_b:.2f}%, {flagged_b.sum():,}건): "
      f"recall={recall_b:.3f}, precision={precision_b:.3f}")
print(f"\n분포 이동(drift) 정도: 의도한 3.00% vs 실제 {actual_pct_b:.2f}% "
      f"(차이 {abs(actual_pct_b - 3.0):.2f}%p)")

test_df['위험등급'] = np.where(test_df['anomaly_score'] >= fixed_threshold, '검토대상', '정상')
print("\n위험등급 분포:")
print(test_df['위험등급'].value_counts())


# 운영 기준 성능 요약 (팀 문서에 반드시 (a)와 함께 기재)
ops_threshold_summary = pd.Series({
    'threshold_percentile': THRESHOLD_PERCENTILE,
    'fixed_threshold_score': fixed_threshold,
    'flagged_pct': actual_pct_b,
    'flagged_count': int(flagged_b.sum()),
    'recall': recall_b,
    'precision': precision_b,
})
print("\n=== (b) 고정 임계값 기준 운영 성능 요약 ===")
print(ops_threshold_summary)


train 기준 고정 임계값(상위 3% 지점): 0.0620

(a) test 내 정확히 top3% (14,097건): recall=0.526, precision=0.611
(b) train 기준 고정 임계값 적용 (실제 3.29%, 15,465건): recall=0.550, precision=0.583

분포 이동(drift) 정도: 의도한 3.00% vs 실제 3.29% (차이 0.29%p)

위험등급 분포:
위험등급
정상      454437
검토대상     15465
Name: count, dtype: int64

=== (b) 고정 임계값 기준 운영 성능 요약 ===
threshold_percentile        97.000000
fixed_threshold_score        0.062046
flagged_pct                  3.291112
flagged_count            15465.000000
recall                       0.550384
precision                    0.583446
dtype: float64


## 8. Pseudo-test 검증 — 시간 기준 분할이면 원래 이 정도 나온다

test PR-AUC(0.5865)가 fold 평균(0.4946)보다 높게 나온 것이 test 쪽 누수 때문이 아닌지 확인하기 위해,
**train 데이터 안에서만** 동일한 시간 기준 분할(2023-01-01 컷오프)을 재현해 pseudo-test를 채점한다.
결과가 fold 평균보다 test에 가깝게 나오면, "시간 기준 분할(카드 이력이 이미 쌓인 상태) 자체가 유리한 조건"이라는
설명이 뒷받침된다.


In [8]:
# train_df 안에서 시간 기준 pseudo-test 분할 (test 구조와 동일하게)
pseudo_cutoff = '2023-01-01'
pseudo_train_idx = train_df.index[train_df['거래일자'] < pseudo_cutoff]
pseudo_valid_idx = train_df.index[train_df['거래일자'] >= pseudo_cutoff]

X_pt = X_train.loc[pseudo_train_idx]
X_pv = X_train.loc[pseudo_valid_idx]
y_pv = train_df.loc[pseudo_valid_idx, '이상거래여부'].values

iso_pseudo = IsolationForest(n_estimators=200, max_samples='auto', contamination='auto', random_state=42, n_jobs=-1)
iso_pseudo.fit(X_pt)
score_pv = -iso_pseudo.decision_function(X_pv)

print("Pseudo-test(train 내부, 시간분할) PR-AUC:", average_precision_score(y_pv, score_pv))

Pseudo-test(train 내부, 시간분할) PR-AUC: 0.553565956852893


## 요약

**이 노트북에서 한 일**
1. `train_processed.csv`(2021~2023) 전체로 최종 Isolation Forest를 학습했다 — 5단계에서 확정한 피처셋
   (Tier0 14개 + 일시불할부구분코드, 그중 모델 직접 입력은 13개 — 거래일자·거래연월은 파생 계산용으로 제외)
   · 하이퍼파라미터(n_estimators=200, max_samples='auto')를 그대로 사용했다.
2. `test_processed.csv`(2024년)에 이 노트북에서 **딱 한 번** 채점했다(v2는 진단 셀 추가를 위한 재실행이며,
   모델 스펙(피처·하이퍼파라미터)은 v1과 동일 — test 점수 자체는 변하지 않았다).
3. 전체 test 성능(PR-AUC, top-k recall/precision)을 확인하고, train 내부 fold 평균(0.4946)과 비교했다.
4. `카드_train노출여부`로 재사용 카드 vs 신규 카드 세그먼트별 성능을 나눠서 확인했다 — 모델이 카드 이력에
   얼마나 의존하는지 정량적으로 드러내기 위함이다.

**해석 시 주의할 점**
- **test PR-AUC(0.5865)는 fold 평균(0.4946)보다 오히려 높게 나왔다.** fold 평균 표준편차(0.051)를 감안해도
  차이가 있어(§5 진단 참고), 이는 성능 저하가 아니라 카드 단위 무작위 fold 분할(보수적 하한선)과 시간 기준
  분할(실제 운영과 동일 조건, 카드 재사용률 97%로 확장 통계가 안정적으로 채워짐)의 **구조적 차이**로 설명된다.
  8절의 pseudo-test(시간 분할, train 내부)가 0.5536으로 test와 유사하게 나와 이 설명을 뒷받침한다.
- 신규 카드 세그먼트는 표본이 매우 작을 수 있어(test 내 약 4,574건, 카드 73개) 지표가 불안정할 수 있다 —
  절대적 수치보다 재사용 카드 대비 상대적 경향을 보는 게 더 안전하다.
- 이 결과를 본 뒤 성능이 부족해 보여도, **이 노트북을 반복 실행해 피처/하이퍼파라미터를 바꾸면 안 된다.**
  개선이 필요하면 5단계(train 내부 fold)로 돌아가 다시 검증한 뒤, 최종 후보가 바뀌었을 때만 새 노트북으로
  다시 한 번 test를 채점해야 한다.

**7절: 우선순위 랭킹 규칙 확정**
- 회계팀 검토 용량을 고려해 상위 3%를 위험 후보로 분류하기로 확정했다. 운영 재현을 위해 train 점수
  분포의 97번째 백분위수를 고정 임계값(0.0620)으로 삼고, test에 적용한 결과 실제 분류 비율 3.29%
  (recall 0.550 / precision 0.583)로 "test 내 정확히 top3%"(recall 0.526 / precision 0.611)와 크게
  다르지 않아, 이 고정 임계값을 운영에 그대로 가져다 써도 안정적이라고 판단한다.

**다음 단계**
- anomaly score(`test_df['anomaly_score']`)를 회계팀 검토 UI용 우선순위 랭킹 리스트로 변환하는 설계
  (7절에서 컷오프는 확정했으므로, 남은 것은 리스트 표시 방식·사유 태깅 등 UI/UX 설계)
- 최종 결과를 팀 문서로 정리하고 Open Issue(#1, #2) 갱신 — Open Issue #1(θ_pass/θ_reject)과 이번 anomaly
  score 컷오프(top3%)는 별개 사안이므로 혼동하지 않도록 명시
- **[운영 전환 시 필요]** `최근7일사용횟수`, `카드누적사용액`, 확장 통계 피처 등은 현재 배치 전처리로
  계산된 값이다. FastMCP `ml_infer` 도구가 실시간 거래 진입 시점에 이 피처들을 어떻게 재현할지
  (Django 내부 read API로 과거 거래 조회 후 계산 vs. 별도 피처 저장소)는 아직 결정되지 않았으므로
  Risk Review Agent 서빙 설계 시 별도 논의가 필요하다.
- fold 4가 다른 fold보다 지속적으로 낮게 나온 원인 — 아직 미조사(참고용)


---
# Part 4 — 운영 컷오프 심화 분석 (결과보고서 §4·§5 재현 코드)

> 이 Part의 두 분석은 결과보고서(`isolation_forest_modeling_결과.md`)에 수치가 기록되어 있으나, 원본 노트북에는
> 코드가 남아있지 않아 **재현 가능한 형태로 여기에 추가**한다. Part 3 실행 직후의 변수
> (`test_anomaly_score`, `y_test`, `test_df`, `train_anomaly_score`)를 그대로 사용한다.
> (이 셀들은 아직 이 통합본에서 실행되지 않았다 — 재실행 시 결과보고서 수치와 일치하는지 확인할 것.)

## 9. recall 목표별 필요 검토 비율

"정상을 이상거래로 보는 것(FP)은 괜찮지만 이상거래를 놓치는 것(FN)은 안 된다"는 요구를 정량화한다.
목표 recall(80/85/90/95/99%)을 달성하려면 점수 상위 몇 %까지 검토해야 하는지, 그때 precision은 얼마인지 계산한다.
결과보고서 §5 기준: recall 90%에는 전체의 32.68% 검토(precision 9.6%), 99%에는 78.72% 검토가 필요했다 —
top3% 컷오프(recall 약 52.6~55.0%)로는 "놓치지 않기" 요구를 충족할 수 없다.


In [ ]:
# Part 3 실행 직후 상태를 가정: test_anomaly_score, y_test, test_df 존재
RECALL_TARGETS = [0.80, 0.85, 0.90, 0.95, 0.99]


def recall_budget_table(y_true, scores, targets=RECALL_TARGETS):
    order = np.argsort(-scores)
    y_sorted = y_true[order]
    cum_tp = np.cumsum(y_sorted)
    n_pos = y_true.sum()
    rows = []
    for t in targets:
        need_tp = int(np.ceil(n_pos * t))
        # 목표 recall을 처음 달성하는 검토 건수 k
        k = int(np.searchsorted(cum_tp, need_tp) + 1)
        rows.append({
            '목표 recall': f'{t:.0%}',
            '필요 검토 건수': k,
            '검토 비율(%)': round(k / len(y_true) * 100, 2),
            '달성 recall': round(cum_tp[k - 1] / n_pos, 4),
            '달성 precision': round(cum_tp[k - 1] / k, 4),
        })
    return pd.DataFrame(rows)


print('=== 전체 test ===')
print(recall_budget_table(y_test, test_anomaly_score))

# 신규 카드 세그먼트 (카드_train노출여부 == False)
_mask_new = ~test_df['카드_train노출여부'].values
print('\n=== 신규 카드 세그먼트 ===')
print(recall_budget_table(y_test[_mask_new], test_anomaly_score[_mask_new]))


## 10. 점수 구간별 실측 이상거래 비율 — raw score의 "확률" 보정 테이블

Isolation Forest의 `s(x,n)` 점수는 보정된 확률(P(이상거래|x))이 아니다 — 정상/이상 그룹의 점수 분포가 크게 겹친다.
따라서 **raw score를 "이상거래일 확률 N%"로 UI·RAG에 그대로 노출하면 안 되며**, test에서 점수 10분위 구간별
실제 관측 이상거래 비율을 계산해 보정 테이블로 쓴다. 결과보고서 §4 기준: 상위 90~100% 구간의 실측 비율은
27.58%(기저율 3.49% 대비 약 8배 lift), 하위 구간은 0.07~1.26% 수준이었다.

**운영 반영 시 주의**
- 상위 10% 구간에 들어가도 실제 확률은 약 27.6% — "위험도 90%" 식 표현은 오도. "이 점수대는 과거 기준 약 X%
  확률로 이상거래였다"는 실측 기반 문구로 구성한다.
- 이 테이블은 모델·피처셋이 바뀌면 재계산해야 하며, RAG에 캐싱 시 **모델 버전과 함께 태깅**한다.
- 절대 확률보다 "상위 몇 % 구간인가"의 상대적 lift로 설명하는 편이 통계적으로 더 정확하다.


In [ ]:
# 점수 10분위(백분위) 구간별 실측 이상거래 비율
decile_edges = np.percentile(test_anomaly_score, np.arange(0, 101, 10))
decile_idx = np.clip(np.searchsorted(decile_edges, test_anomaly_score, side='right') - 1, 0, 9)

calib = (
    pd.DataFrame({'구간': decile_idx, 'y': y_test})
    .groupby('구간')['y']
    .agg(건수='size', 실측이상거래비율='mean')
)
calib['실측이상거래비율(%)'] = (calib['실측이상거래비율'] * 100).round(2)
calib['기저율대비_lift'] = (calib['실측이상거래비율'] / y_test.mean()).round(2)
calib.index = [f'{i*10}~{(i+1)*10}%' for i in range(10)]

print(f'전체 기저율: {y_test.mean()*100:.2f}%')
calib[['건수', '실측이상거래비율(%)', '기저율대비_lift']]


---
# 종합 요약 — 확정 사항과 미해결 사안

**확정 (Open Issue #2 관련 — 1차 해소)**
- 알고리즘: **Isolation Forest** (`n_estimators=200`, `max_samples='auto'`, `contamination='auto'`) — 하이퍼파라미터
  튜닝 개선폭이 fold 노이즈(σ≈0.051) 이하라 베이스라인 유지
- 피처셋: **Tier0(14개) + `일시불할부구분코드` = 15개** (모델 직접 입력 13개, 원-핫 후 24컬럼)
- 검증 체계: 카드 단위 StratifiedGroupKFold 5-fold(보수적 하한선, 평균 PR-AUC 0.4946) + 시간 기준 최종 test
  (PR-AUC 0.5865) + pseudo-test(0.5536)로 test 수치의 타당성 교차 확인
- 지도학습 미사용: `이상거래여부`는 평가 전용, `decision_labels`는 축적만 — MVP 문서 지침과 일치

**미확정 / 재검토 필요**
- **운영 컷오프**: top3% 고정 임계값(0.0620)은 임계값 안정성(드리프트 0.29%p)은 확인됐으나, recall이 55.0%에
  그쳐 "이상거래를 놓치면 안 된다"는 원칙과 충돌 — recall 90%에는 전체의 약 33% 검토 필요(Part 4 §9).
  → 1단계 컷오프를 넉넉히(top10~15%) 잡고 2단계(RAG 내규 검증)로 정밀도를 보강하는 방향을 PM·회계팀과 논의
- **recall 목표치 합의**: "절대 놓치면 안 된다"를 몇 %로 정량화할지, 그에 따른 검토 용량 증가를 감당 가능한지
- fold 4가 지속적으로 낮게 나온 원인 미조사
- **[운영 전환 시]** `최근7일사용횟수`·확장 통계 등 배치 계산 피처를 FastMCP `ml_infer` 실시간 진입 시점에
  어떻게 재현할지(Django 내부 read API 조회 후 계산 vs 별도 피처 저장소) — Risk Review Agent 서빙 설계 시 논의
- Open Issue #1(Rule Agent의 θ_pass/θ_reject)과 이번 anomaly score 컷오프는 **별개 사안** — 혼동 주의
